In [0]:
%pip install ddgs -q

In [0]:
%pip install ddgs bs4 lxml requests feedparser duckduckgo_search -q

In [0]:
dbutils.library.restartPython()

In [0]:
# ============================================================
# US IT JOB HARVESTER V8 — TARGETED JOB BOARDS + SMART SNIPPET
# ============================================================
# FIXES vs V7:
#   🔴 ROOT CAUSE 1 FIXED: Portals updated to EXACTLY what you asked:
#      LinkedIn, Indeed, Dice, Glassdoor, Wellfound, Built In, ZipRecruiter.
#   🔴 ROOT CAUSE 2 FIXED: Validation score is now REALISTIC (min 35).
#      Job boards rarely show salary/emails. Focus is on Title + Description.
#   🔴 ROOT CAUSE 3 FIXED: "Snippet Trust Strategy". We DO NOT scrape 
#      LinkedIn/Indeed directly (avoids 403/Cloudflare). We trust the 
#      DuckDuckGo search snippet, which is fast and 100% reliable.
#   ✅ NEW: Strict "last 24 hours" filter for yesterday/today jobs.
# ============================================================
# ONE-TIME SETUP: %pip install duckduckgo-search

import hashlib
import uuid
import re
import time
import random
import logging
from datetime import date, datetime
from duckduckgo_search import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, TimestampType, IntegerType
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# ============================================================
# ⚙️  CONFIG
# ============================================================
CONFIG = {
    "delta_table": "main.jobs_automation.raw_jobs_staging", # మీ actual table path పెట్టుకోండి
    "max_results": 15,
    "time_filter": "d",       # 'd' = last 24 hours (yesterday + today)
    "min_score": 35,          # Realistic threshold for job boards
    "search_delay": (2.0, 4.0),
}

# ============================================================
# 🎯 TARGET PORTALS (Exactly what you requested)
# ============================================================
PORTAL_SITES = [
    "site:linkedin.com/jobs",
    "site:indeed.com",
    "site:dice.com",
    "site:glassdoor.com",
    "site:wellfound.com",
    "site:builtin.com",
    "site:ziprecruiter.com",
]

# These sites block direct scraping. We TRUST their search snippets instead.
SNIPPET_TRUSTED_SITES = {
    "linkedin.com", "indeed.com", "glassdoor.com", 
    "ziprecruiter.com", "dice.com", "wellfound.com", "builtin.com"
}

ROLES = [
    "Data Engineer", "Senior Data Engineer",
    "PySpark Engineer", "Spark Developer",
    "ETL Developer", "Data Pipeline Engineer",
    "Analytics Engineer", "BI Engineer",
    "Python Developer", "Python Engineer",
    "Machine Learning Engineer", "MLOps Engineer"
]

GARBAGE_DOMAINS = {"youtube.com", "github.com", "stackoverflow.com", "medium.com", "reddit.com", "quora.com", "udemy.com", "coursera.org"}

# ============================================================
# SECTION 1: SMART VALIDATORS (Realistic for Job Boards)
# ============================================================
def is_garbage_url(url: str) -> bool:
    return any(d in url.lower() for d in GARBAGE_DOMAINS)

def is_real_job_page(text: str) -> bool:
    if not text or len(text) < 80:
        return False
    low = text.lower()
    # Must have at least 2 core job keywords
    job_keywords = ["experience", "skills", "requirements", "responsibilities", "role", "candidate", "apply", "qualifications", "hiring"]
    return sum(1 for kw in job_keywords if kw in low) >= 2

def compute_validation_score(rec: dict) -> int:
    score = 0
    desc  = rec.get("job_description", "").lower()
    title = rec.get("job_title", "").lower()
    company = rec.get("company_name", "").lower()

    # 1. Title is meaningful (30 pts)
    if len(title) > 5 and "unknown" not in title and "job" not in title:
        score += 30

    # 2. Company is meaningful (20 pts)
    if len(company) > 3 and "unknown" not in company and "web extracted" not in company:
        score += 20

    # 3. Description has job keywords (30 pts)
    kw_count = sum(1 for kw in ["experience", "skills", "requirements", "responsibilities"] if kw in desc)
    if kw_count >= 2: score += 30
    elif kw_count >= 1: score += 15

    # 4. Description length is decent (20 pts)
    if len(desc) > 300: score += 20
    elif len(desc) > 100: score += 10

    return min(score, 100)

# ============================================================
# SECTION 2: TEXT CLEANERS
# ============================================================
def clean_title(raw: str) -> str:
    s = re.sub(r"\[.*?\]|\(.*?\)", "", raw)
    s = re.sub(r"(?i)(Job Application for|Apply for|Jobs In)\s+", "", s)
    for suffix in [" - LinkedIn", " | LinkedIn", " - Indeed", " | Glassdoor", " - ZipRecruiter", " | Wellfound"]:
        s = re.sub(rf"(?i){suffix}.*$", "", s)
    return s.strip()

def split_title_and_company(raw_title: str):
    cleaned = clean_title(raw_title)
    job_title, company = cleaned, "Unknown"
    for sep in [" at ", " At ", " - ", " | ", " @ "]:
        if sep in cleaned:
            parts = cleaned.split(sep, 1)
            job_title, company = parts[0].strip(), parts[1].strip()
            break
    
    job_title = re.sub(r"[^a-zA-Z0-9\s\+\#\.]", "", job_title).strip().title()
    company = re.sub(r"[^a-zA-Z0-9\s\.,&\-]", "", company).strip().title()
    return (job_title or "Unknown"), (company or "Unknown")

# ============================================================
# SECTION 3: RESULT PROCESSOR (Snippet Trust Strategy)
# ============================================================
def process_result(res: dict, canonical_role: str) -> dict | None:
    url = (res.get("href") or "").strip()
    snippet = (res.get("body") or "").strip()
    s_title = (res.get("title") or "").strip()

    if not url or is_garbage_url(url):
        return None

    # 🌟 SMART FALLBACK: If it's LinkedIn/Indeed, TRUST the snippet. 
    # Do NOT attempt to scrape (avoids 403/Cloudflare and saves 20 mins).
    is_trusted = any(site in url.lower() for site in SNIPPET_TRUSTED_SITES)
    
    if is_trusted:
        page_title = s_title
        description = snippet
        was_scraped = False
    else:
        # For other sites, you could add a lightweight requests.get here if needed
        # For now, fallback to snippet to keep it ultra-fast and reliable
        page_title = s_title
        description = snippet
        was_scraped = False

    if not is_real_job_page(description):
        return None

    job_title, company = split_title_and_company(page_title)
    if len(job_title) < 3:
        job_title = canonical_role.title()

    now = datetime.now()
    rec = {
        "id":                  str(uuid.uuid4()),
        "job_hash":            hashlib.md5(url.encode()).hexdigest(),
        "created_at":          now,
        "updated_at":          now,
        "fetch_date":          now.date(),
        "search_keyword":      canonical_role,
        "company_name":        company,
        "job_title":           job_title,
        "job_description":     description,
        "apply_link":          url,
        "hr_email":            "Not Found", # Job boards rarely show this
        "job_type":            "Remote" if "remote" in description.lower() else "Not Specified",
        "salary_range":        "Not Specified", # Job boards rarely show this in snippet
        "experience_required": "Not Specified",
        "location":            "USA",
        "skills_required":     "Not Specified",
        "link_status":         "Active",
        "validation_score":    0,
        "validation_status":   "Pending",
    }
    
    score = compute_validation_score(rec)
    rec["validation_score"] = score
    rec["validation_status"] = "Valid" if score >= 70 else "Partial" if score >= CONFIG["min_score"] else "Junk"

    if score < CONFIG["min_score"]:
        return None

    return rec

# ============================================================
# SECTION 4: SEARCH ENGINE
# ============================================================
def search_for_role(canonical_role: str) -> list:
    records = []
    seen_urls = set()
    
    # Format role for exact match search
    phrase = f'"{canonical_role}"'
    
    for portal in PORTAL_SITES:
        # Optimal query format for DuckDuckGo: "Role" site:portal.com
        query = f"{phrase} {portal}"
        log.info(f"🔍 Searching: {query}")
        
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(
                    query,
                    region="us-en",
                    max_results=CONFIG["max_results"],
                    timelimit=CONFIG["time_filter"] # Last 24 hours!
                ))
            
            if not results:
                continue
                
            log.info(f"  📥 Found {len(results)} links. Processing...")
            
            for res in results:
                href = res.get("href", "")
                if href in seen_urls:
                    continue
                seen_urls.add(href)
                
                job = process_result(res, canonical_role)
                if job:
                    records.append(job)
                    
            time.sleep(random.uniform(*CONFIG["search_delay"]))
            
        except Exception as e:
            log.warning(f"  ⚠️ Search failed for {portal}: {str(e)[:50]}")
            time.sleep(3)

    log.info(f"  ✅ {canonical_role} → {len(records)} valid jobs found")
    return records

# ============================================================
# SECTION 5: DELTA LAKE WRITER
# ============================================================
JOB_SCHEMA = StructType([
    StructField("id", StringType(), True),
    StructField("job_hash", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("fetch_date", DateType(), True),
    StructField("search_keyword", StringType(), True),
    StructField("company_name", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("job_description", StringType(), True),
    StructField("apply_link", StringType(), True),
    StructField("hr_email", StringType(), True),
    StructField("job_type", StringType(), True),
    StructField("salary_range", StringType(), True),
    StructField("experience_required", StringType(), True),
    StructField("location", StringType(), True),
    StructField("skills_required", StringType(), True),
    StructField("link_status", StringType(), True),
    StructField("validation_score", IntegerType(), True),
    StructField("validation_status", StringType(), True),
])

def write_to_delta(records: list, spark: SparkSession) -> int:
    if not records:
        return 0
    
    table = CONFIG["delta_table"]
    df = spark.createDataFrame(records, schema=JOB_SCHEMA).dropDuplicates(["job_hash"])
    
    # Create table if not exists (simplified)
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id STRING, job_hash STRING, created_at TIMESTAMP, updated_at TIMESTAMP,
            fetch_date DATE, search_keyword STRING, company_name STRING, job_title STRING,
            job_description STRING, apply_link STRING, hr_email STRING, job_type STRING,
            salary_range STRING, experience_required STRING, location STRING,
            skills_required STRING, link_status STRING, validation_score INT, validation_status STRING
        ) USING DELTA PARTITIONED BY (fetch_date)
    """)
    
    count = df.count()
    if count > 0:
        (DeltaTable.forName(spark, table).alias("tgt")
         .merge(df.alias("src"), "tgt.job_hash = src.job_hash")
         .whenNotMatchedInsertAll()
         .execute())
    return count

# ============================================================
# ▶️  MAIN ORCHESTRATOR
# ============================================================
def run_harvester_v8():
    log.info("=" * 65)
    log.info("🚀 US IT JOB HARVESTER V8 — SMART SNIPPET STRATEGY")
    log.info("🎯 Target: LinkedIn, Indeed, Dice, Glassdoor, Wellfound, Built In, ZipRecruiter")
    log.info("⏱️  Filter: Last 24 Hours (Yesterday + Today)")
    log.info("=" * 65)

    all_records = []
    for role in ROLES:
        try:
            jobs = search_for_role(role)
            all_records.extend(jobs)
        except Exception as e:
            log.error(f"❌ Error for {role}: {e}")
        time.sleep(random.uniform(2.0, 4.0))

    # Global Dedup
    seen, unique = set(), []
    for rec in all_records:
        if rec["job_hash"] not in seen:
            seen.add(rec["job_hash"])
            unique.append(rec)

    log.info(f"\n📊 Total Raw: {len(all_records)} | After Dedup: {len(unique)}")
    
    written = write_to_delta(unique, spark)
    log.info(f"🎉 HARVEST COMPLETE — {written} new records written to Delta!")

    # Show Summary
    try:
        spark.sql(f"""
            SELECT search_keyword, validation_status, COUNT(*) as jobs, 
                   ROUND(AVG(validation_score)) as avg_score
            FROM {CONFIG['delta_table']}
            WHERE fetch_date = current_date()
            GROUP BY search_keyword, validation_status
            ORDER BY jobs DESC
        """).show(20, truncate=False)
    except Exception:
        pass

# 🚀 RUN IT
run_harvester_v8()

In [0]:
# ============================================================
# US IT JOB HARVESTER V8 — JSEARCH API + ATS DIRECT SCRAPE
# ============================================================
# ROOT CAUSE FIX (Why V7 returned garbage links):
#   ❌ DuckDuckGo site:linkedin.com → LinkedIn blocks bots
#      → DDGS returns random tutorial pages (gardenerspath, pypi, etc.)
#   ❌ site:indeed.com, site:dice.com → same problem
#   ❌ 237-char OR queries confuse all search engines
#
# V8 ARCHITECTURE:
#   ✅ LAYER 1 — JSearch API (RapidAPI): LinkedIn + Indeed + Glassdoor
#               + Dice + ZipRecruiter + Wellfound — ONE API, all portals
#   ✅ LAYER 2 — Direct ATS scraping: Greenhouse + Lever + Ashby + Workable
#               (these have no bot protection, DDGS site: works fine)
#   ✅ LAYER 3 — Dice RSS feed (free, no auth needed)
#   ✅ Content validation score 0-100 (same as V7)
#   ✅ Delta MERGE dedup on job_hash
#   ✅ All jobs written to same Delta table as before
#
# ONE-TIME SETUP:
#   %pip install curl_cffi duckduckgo-search feedparser
# ============================================================

import hashlib
import uuid
import re
import time
import random
import feedparser
from bs4 import BeautifulSoup
from datetime import date, datetime
from duckduckgo_search import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DateType, TimestampType, IntegerType,
)
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutureTimeout
import warnings
import logging

try:
    from curl_cffi import requests as cf_requests
    _HAS_CURL_CFFI = True
except ImportError:
    import requests as cf_requests
    _HAS_CURL_CFFI = False

import requests as std_requests   # always available for JSearch API calls

warnings.filterwarnings("ignore", category=ResourceWarning)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ============================================================
# ⚙️  CONFIG
# ============================================================
CONFIG = {
    # Delta table (Unity Catalog)
    "delta_table": "main.jobs_automation.raw_jobs_staging",

    # JSearch API key — stored in Databricks secrets
    # Setup: dbutils.secrets.put("jobs_automation", "jsearch", "<your_key>")
    "jsearch_secret_scope": "jobs_automation",
    "jsearch_secret_key":   "jsearch",

    # JSearch pages per keyword (10 results/page → pages=5 → 50 jobs/keyword)
    "jsearch_pages": 5,

    # ATS scraping via DDGS (Greenhouse, Lever, Ashby, Workable)
    "ddgs_max_results": 15,
    "ddgs_time_filter": "d",          # 'd'=last day, 'w'=last week

    # Dice RSS date filter: 'r1'=last day, 'r7'=last week
    "dice_date_filter": "r1",

    # Concurrent scrape workers for ATS pages
    "scrape_workers": 4,

    # HTTP timeout
    "http_timeout": 12,

    # Minimum validation score to keep (0-100)
    "min_score": 45,

    # Delay between DDGS queries
    "search_delay": (5.0, 8.0),

    # Sleep on rate-limit
    "ratelimit_sleep": (15.0, 25.0),
}

# ============================================================
# ROLES — search terms for each layer
# ============================================================
ROLES = {
    "Data Engineer":             ['"Data Engineer"', '"Senior Data Engineer"'],
    "Spark / PySpark Engineer":  ['"PySpark Engineer"', '"Spark Engineer"'],
    "ETL Developer":             ['"ETL Developer"', '"ETL Engineer"'],
    "Analytics Engineer":        ['"Analytics Engineer"', '"BI Engineer"'],
    "Data Platform Engineer":    ['"Data Platform Engineer"'],
    "Cloud Data Engineer":       ['"Cloud Data Engineer"', '"AWS Data Engineer"'],
    "Databricks Engineer":       ['"Databricks Engineer"', '"Databricks Developer"'],
    "Python Developer":          ['"Python Developer"', '"Python Engineer"'],
    "Machine Learning Engineer": ['"Machine Learning Engineer"', '"MLOps Engineer"'],
}

# ============================================================
# ATS PORTALS — where DDGS site: search works reliably
# LinkedIn/Indeed/Dice NOT here — use JSearch API instead
# ============================================================
ATS_PORTAL_SITES = [
    "site:boards.greenhouse.io",
    "site:jobs.lever.co",
    "site:jobs.ashbyhq.com",
    "site:apply.workable.com",
    "site:myworkdayjobs.com",
    "site:smartrecruiters.com/jobs",
    "site:weworkremotely.com/remote-jobs",
    "site:remoteok.com",
]

# Dice RSS feeds (role slug → search URL)
DICE_RSS_ROLES = {
    "Data Engineer":             "data+engineer",
    "Python Developer":          "python+developer",
    "ETL Developer":             "etl+developer",
    "Machine Learning Engineer": "machine+learning+engineer",
    "Analytics Engineer":        "analytics+engineer",
    "Databricks Engineer":       "databricks+engineer",
}

GOOD_BACKENDS = ["duckduckgo", "startpage", "brave"]

SNIPPET_ONLY_DOMAINS: set = {
    "linkedin.com", "indeed.com", "dice.com", "glassdoor.com",
    "ziprecruiter.com", "monster.com", "wellfound.com",
    "careerbuilder.com",
}

GARBAGE_DOMAINS: set = {
    "wikipedia.org", "w3schools.com", "python.org", "geeksforgeeks.org",
    "tutorialspoint.com", "ibm.com/docs", "ibm.com/think",
    "learn.microsoft.com", "docs.databricks.com", "spark.apache.org",
    "youtube.com", "stackoverflow.com", "medium.com", "dev.to",
    "reddit.com", "quora.com", "coursera.org", "udemy.com",
    "data.gov", "cdc.gov", "va.gov", "nih.gov", "census.gov",
    "espn.com", "mathsisfun.com", "mygreatlearning.com",
    "gardenerspath.com", "pypi.org", "oracle.com/integration",
    "informatica.com/resources", "sas.com/insights",
    "ml-ops.org", "mechlesson.com", "kodekloud.com",
    "theengineeringchoice.org", "community.databricks.com",
    "bing.com", "google.com", "aws.amazon.com/products",
    "drive4spark", "sparkdriverapp",
}

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/130.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/131.0.0.0 Safari/537.36",
]

# ============================================================
# SECTION 1: URL FILTERS
# ============================================================

def is_garbage_url(url: str) -> bool:
    low = url.lower()
    return any(d in low for d in GARBAGE_DOMAINS)

def is_snippet_only(url: str) -> bool:
    low = url.lower()
    return any(d in low for d in SNIPPET_ONLY_DOMAINS)

# ============================================================
# SECTION 2: CONTENT VALIDATORS
# ============================================================

_JOB_KEYWORDS = [
    "engineer", "developer", "position", "experience", "skills",
    "requirements", "responsibilities", "hiring", "apply",
    "qualifications", "remote", "full-time", "salary", "benefits",
    "team", "role", "candidate", "opportunity", "employment",
]
_SPANISH_SIGNALS = ["años", "quinceanera", "quinceañera", "fiesta", "viaje", "celebra"]
_GERMAN_SIGNALS  = ["gmbh", "gründen", "buchhaltung", "kostenlos"]

def is_real_job_page(text: str) -> bool:
    if not text or len(text) < 120:
        return False
    low = text.lower()
    if sum(1 for s in _SPANISH_SIGNALS if s in low) >= 2:
        return False
    if sum(1 for s in _GERMAN_SIGNALS if s in low) >= 2:
        return False
    return sum(1 for kw in _JOB_KEYWORDS if kw in low) >= 3

def compute_validation_score(rec: dict) -> int:
    score = 0
    desc  = rec.get("job_description", "")
    dlen  = len(desc)

    if is_real_job_page(desc):           score += 25
    if dlen > 1500:                      score += 20
    elif dlen > 500:                     score += 12
    elif dlen > 200:                     score += 5

    if rec.get("company_name", "Unknown") not in ("Unknown", "", "Not Specified"):
        score += 15
    if rec.get("job_type", "Not Specified") != "Not Specified":
        score += 10
    if rec.get("skills_required", "Not Specified") != "Not Specified":
        score += 10
    if rec.get("salary_range", "Not Specified") != "Not Specified":
        score += 10
    if rec.get("hr_email", "Not Found") != "Not Found":
        score += 5
    if rec.get("experience_required", "Not Specified") != "Not Specified":
        score += 5
    return min(score, 100)

def get_validation_status(score: int) -> str:
    if score >= 70:   return "Valid"
    if score >= CONFIG["min_score"]: return "Partial"
    return "Junk"

# ============================================================
# SECTION 3: TEXT CLEANERS & EXTRACTORS
# ============================================================

_SPAM_SUFFIXES = [
    r" - LinkedIn", r" \| LinkedIn", r" - Greenhouse", r" - Lever",
    r" \| Wellfound", r" - Dice\.com", r" \| Built In", r" - Indeed",
    r" - ZipRecruiter", r" \| Glassdoor", r" - Monster",
    r" - Remote OK", r" - Jobvite", r" \| Workable",
    r" - SmartRecruiters", r" - Ashby",
]

def _clean_raw_title(raw: str) -> str:
    s = re.sub(r"\[.*?\]|\(.*?\)", "", raw)
    s = re.sub(r"(?i)(Job Application for|Apply for|Jobs In)\s+", "", s)
    for sfx in _SPAM_SUFFIXES:
        s = re.sub(rf"(?i){sfx}.*$", "", s)
    return s.strip()

def split_title_and_company(raw_title: str):
    cleaned   = _clean_raw_title(raw_title)
    job_title = cleaned
    company   = "Unknown"
    for sep in [" at ", " At ", " - ", " | ", " @ ", " – "]:
        if sep in cleaned:
            parts     = cleaned.split(sep, 1)
            job_title = parts[0].strip()
            company   = parts[1].strip()
            break
    job_title = re.sub(r"[^a-zA-Z0-9\s\+\#\./,]", "", job_title).strip().title()
    company   = re.sub(r"[^a-zA-Z0-9\s\.,&\-]",   "", company  ).strip().title()
    return (job_title or "Unknown"), (company or "Unknown")

def extract_emails(text: str) -> str:
    found = re.findall(r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,7}\b", text)
    _noise = {"noreply","no-reply","donotreply","notifications","mailer","bounce",
              "alert","privacy","legal","unsubscribe","sentry","example.com",
              "support","help","feedback","abuse","postmaster"}
    clean = [e for e in found if not any(n in e.lower() for n in _noise) and len(e) > 6]
    return ", ".join(sorted(set(clean))) if clean else "Not Found"

def detect_job_type(desc: str) -> str:
    d = desc.lower(); tags = []
    if re.search(r"\bc2c\b|corp[\s\-]?to[\s\-]?corp", d):           tags.append("C2C")
    if re.search(r"\bw[\s\-]?2\b", d):                               tags.append("W2")
    if re.search(r"\b1099\b", d):                                    tags.append("1099")
    if re.search(r"\bcontract[\s\-]to[\s\-]hire\b|\bc2h\b", d):     tags.append("Contract-to-Hire")
    elif re.search(r"\bcontract\b|\bcontractor\b", d):               tags.append("Contract")
    if re.search(r"\bfull[\s\-]?time\b|\bpermanent\b|\bfte\b", d):  tags.append("Full-Time")
    if re.search(r"\bpart[\s\-]?time\b", d):                        tags.append("Part-Time")
    if re.search(r"\bremote\b|work[\s\-]from[\s\-]home|\bwfh\b", d): tags.append("Remote")
    if re.search(r"\bhybrid\b", d):                                  tags.append("Hybrid")
    if re.search(r"\bon[\s\-]?site\b|in[\s\-]?office\b", d):        tags.append("Onsite")
    return ", ".join(tags) if tags else "Not Specified"

def extract_salary(desc: str) -> str:
    patterns = [
        r"\$[\d,]+(?:\.\d{2})?\s*[-–to]+\s*\$[\d,]+(?:\.\d{2})?(?:\s*(?:k|K|/hr|/hour|/year|/yr|annually))?",
        r"\$[\d,]+(?:\.\d{2})?(?:\s*(?:k|K|/hr|/hour|/year|/yr|annually))",
        r"[\d,]+\s*[-–]\s*[\d,]+\s*(?:USD|k|K)\s*(?:per\s+(?:year|hr|hour)|annually|/hr)",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m: return m.group(0).strip()
    return "Not Specified"

def extract_experience(desc: str) -> str:
    patterns = [
        r"\d+\+?\s*[-–to]+\s*\d+\s+years?\s+(?:of\s+)?(?:professional\s+)?experience",
        r"\d+\+?\s+years?\s+(?:of\s+)?(?:professional\s+)?experience",
        r"(?:minimum|min\.?|at\s+least)\s+\d+\+?\s+years?",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m: return m.group(0).strip()
    return "Not Specified"

def extract_location(desc: str) -> str:
    _states = ("AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|"
               "MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|"
               "TX|UT|VT|VA|WA|WV|WI|WY|DC")
    patterns = [
        rf"(?:location|located\s+in|based\s+in|office\s+in)\s*[:\-]?\s*([A-Za-z\s]+,?\s*(?:{_states}|USA|United\s+States))",
        rf"([A-Za-z][A-Za-z\s]+,\s*(?:{_states}))\b",
        r"(?:Remote|Hybrid|Onsite)[,\s]+(?:in\s+)?([A-Za-z\s,]{4,40})",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m:
            loc = m.group(1).strip().rstrip(",").strip()
            if 2 < len(loc) < 60: return loc
    return "Not Specified"

_SKILLS = [
    (r"\bPython\b","Python"),(r"\bScala\b","Scala"),(r"\bJava\b","Java"),
    (r"\b(?:SQL|T-SQL|PL/SQL)\b","SQL"),(r"\bPySpark\b","PySpark"),
    (r"\b(?:Apache\s+)?Spark\b","Spark"),(r"\b(?:Apache\s+)?Kafka\b","Kafka"),
    (r"\b(?:Apache\s+)?Flink\b","Flink"),(r"\b(?:Apache\s+)?Hadoop\b","Hadoop"),
    (r"\bHive\b","Hive"),(r"\bKinesis\b","Kinesis"),
    (r"\bAWS\b|Amazon\s+Web\s+Services","AWS"),(r"\bAzure\b|Microsoft\s+Azure","Azure"),
    (r"\bGCP\b|Google\s+Cloud","GCP"),(r"\bDatabricks\b","Databricks"),
    (r"\bSnowflake\b","Snowflake"),(r"\bRedshift\b","Redshift"),
    (r"\bBigQuery\b","BigQuery"),(r"\bDelta\s+Lake\b","Delta Lake"),
    (r"\bIceberg\b","Iceberg"),(r"\bAirflow\b","Airflow"),
    (r"\bdbt\b|data\s+build\s+tool","dbt"),(r"\bTerraform\b","Terraform"),
    (r"\bDocker\b","Docker"),(r"\bKubernetes\b|\bK8s\b","Kubernetes"),
    (r"\bPostgreSQL\b|\bpostgres\b","PostgreSQL"),(r"\bMySQL\b","MySQL"),
    (r"\bMongoDB\b","MongoDB"),(r"\bMLflow\b","MLflow"),
    (r"\bPower\s+BI\b","Power BI"),(r"\bTableau\b","Tableau"),(r"\bLooker\b","Looker"),
]

def extract_skills(desc: str) -> str:
    found = []
    for pattern, label in _SKILLS:
        if re.search(pattern, desc, re.IGNORECASE) and label not in found:
            found.append(label)
    return ", ".join(found[:30]) if found else "Not Specified"

# ============================================================
# SECTION 4: RECORD BUILDER (shared by all layers)
# ============================================================

_EXPIRED_SIGNALS = [
    "job no longer available", "position has been filled",
    "this job has expired", "no longer accepting applications",
    "job has been closed", "listing has expired",
]

def build_record(
    url: str,
    description: str,
    raw_title: str,
    canonical_role: str,
    source: str = "unknown",
    was_scraped: bool = True,
) -> dict | None:
    """Build + validate one job record. Returns None if below threshold."""
    if not url or is_garbage_url(url):
        return None
    if not is_real_job_page(description):
        log.debug("⛔ Not a job page: %s", url[:70])
        return None

    low = description.lower()
    if any(s in low for s in _EXPIRED_SIGNALS):
        log.info("⏩ Expired: %s", url[:70])
        return None

    job_title, company = split_title_and_company(raw_title)
    if not job_title or len(job_title.strip()) < 2:
        job_title = canonical_role.title()

    link_status = "Active" if was_scraped else "Unverified"
    now = datetime.now()

    rec = {
        "id":                  str(uuid.uuid4()),
        "job_hash":            hashlib.md5(url.encode()).hexdigest(),
        "created_at":          now,
        "updated_at":          now,
        "fetch_date":          now.date(),
        "search_keyword":      canonical_role,
        "source":              source,
        "company_name":        company,
        "job_title":           job_title,
        "job_description":     description,
        "apply_link":          url,
        "hr_email":            extract_emails(description),
        "job_type":            detect_job_type(description),
        "salary_range":        extract_salary(description),
        "experience_required": extract_experience(description),
        "location":            extract_location(description),
        "skills_required":     extract_skills(description),
        "link_status":         link_status,
        "validation_score":    0,
        "validation_status":   "Pending",
    }
    score                   = compute_validation_score(rec)
    rec["validation_score"] = score
    rec["validation_status"]= get_validation_status(score)

    if score < CONFIG["min_score"]:
        log.debug("⛔ Score %d < %d: %s", score, CONFIG["min_score"], url[:70])
        return None
    return rec

# ============================================================
# SECTION 5: LAYER 1 — JSEARCH API
# Covers: LinkedIn, Indeed, Glassdoor, Dice, ZipRecruiter, Wellfound
# ============================================================

def _get_jsearch_key() -> str:
    try:
        return dbutils.secrets.get(
            CONFIG["jsearch_secret_scope"],
            CONFIG["jsearch_secret_key"],
        )
    except Exception:
        raise RuntimeError(
            "❌ JSearch API key not found in Databricks secrets.\n"
            "   Run: dbutils.secrets.put('jobs_automation', 'jsearch', '<your_key>')"
        )

def fetch_jsearch_jobs(role_label: str, search_phrase: str) -> list:
    """
    Call JSearch API for one role. Returns list of validated records.
    JSearch covers: LinkedIn, Indeed, Glassdoor, Dice, ZipRecruiter, Wellfound.
    """
    try:
        api_key = _get_jsearch_key()
    except RuntimeError as e:
        log.error(str(e))
        return []

    url     = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key":  api_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com",
    }

    all_jobs: list = []

    for page in range(1, CONFIG["jsearch_pages"] + 1):
        params = {
            "query":          f"{search_phrase} USA",
            "page":           str(page),
            "num_pages":      "1",
            "date_posted":    "today",   # only today's jobs!
            "employment_types": "FULLTIME,CONTRACTOR,PARTTIME",
        }
        try:
            resp = std_requests.get(url, headers=headers, params=params, timeout=20)
            if resp.status_code == 429:
                log.warning("  ⚠️  JSearch rate limit — sleeping 30s")
                time.sleep(30)
                continue
            if resp.status_code != 200:
                log.warning("  JSearch HTTP %d for page %d", resp.status_code, page)
                break

            data     = resp.json()
            jobs_raw = data.get("data", [])
            if not jobs_raw:
                break   # no more pages

            log.info("  📥 JSearch page %d → %d jobs for '%s'", page, len(jobs_raw), search_phrase)

            for job in jobs_raw:
                if not isinstance(job, dict):
                    continue

                desc   = job.get("job_description", "") or ""
                title  = job.get("job_title", "")        or ""
                company= job.get("employer_name", "")    or "Unknown"
                link   = (
                    job.get("job_apply_link")
                    or job.get("job_google_link")
                    or ""
                )
                if not link:
                    continue

                raw_title = f"{title} at {company}" if company and company != "Unknown" else title

                # Enrich description with structured fields JSearch provides
                extra = []
                if job.get("job_city"):       extra.append(f"Location: {job['job_city']}, {job.get('job_state','')}")
                if job.get("job_min_salary"): extra.append(f"Salary: ${job['job_min_salary']:,} - ${job.get('job_max_salary', job['job_min_salary']):,} {job.get('job_salary_period','')}")
                if job.get("job_employment_type"): extra.append(f"Type: {job['job_employment_type']}")
                if job.get("job_required_skills"):  extra.append(f"Skills: {', '.join(job['job_required_skills'])}")
                if extra:
                    desc = "\n".join(extra) + "\n\n" + desc

                rec = build_record(
                    url=link,
                    description=desc[:8000],
                    raw_title=raw_title,
                    canonical_role=role_label,
                    source="jsearch",
                    was_scraped=True,
                )
                if rec:
                    all_jobs.append(rec)

            time.sleep(random.uniform(1.0, 2.0))   # polite gap between pages

        except Exception as exc:
            log.error("  JSearch error page %d: %s", page, str(exc)[:80])
            break

    log.info("  ✅ JSearch → %d valid jobs for '%s'", len(all_jobs), role_label)
    return all_jobs

# ============================================================
# SECTION 6: LAYER 2 — DIRECT ATS SCRAPING (Greenhouse/Lever/etc.)
# Only for portals where DDGS site: search actually works
# ============================================================

def _make_session():
    if _HAS_CURL_CFFI:
        return cf_requests.Session(impersonate="chrome131")
    import requests as _r
    s = _r.Session()
    s.headers.update({"User-Agent": random.choice(USER_AGENTS)})
    return s

_SCRAPE_SESSION = _make_session()

_CONTENT_IDS      = ["content","job-description","jobDescription","job_description","main-content"]
_CONTENT_CLASSES  = [re.compile(p, re.IGNORECASE) for p in [
    r"job[\-_]?description", r"job[\-_]?details", r"\bdescription\b", r"\bcontent\b",
]]
_BOT_SIGNALS      = ["checking your browser","access denied","cloudflare","captcha","enable javascript","just a moment"]

def _extract_main_content(soup: BeautifulSoup) -> str:
    for id_val in _CONTENT_IDS:
        el = soup.find(id=id_val)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    for pat in _CONTENT_CLASSES:
        el = soup.find(class_=pat)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    for tag in ["main","article"]:
        el = soup.find(tag)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    return soup.get_text(separator=" ", strip=True)

def scrape_ats_url(url: str, fallback_snippet: str = "") -> tuple:
    """Returns (page_title, description, was_scraped)."""
    if is_snippet_only(url):
        return "", fallback_snippet, False
    for _ in range(2):
        try:
            resp = _SCRAPE_SESSION.get(url, timeout=CONFIG["http_timeout"], allow_redirects=True)
            if resp.status_code in (403, 429, 503):
                time.sleep(random.uniform(4, 8))
                continue
            if resp.status_code != 200:
                return "", fallback_snippet, False
            soup = BeautifulSoup(resp.text, "lxml")
            for tag in soup(["script","style","footer","nav","header","aside","noscript","iframe"]):
                tag.decompose()
            page_title  = soup.title.string.strip() if soup.title and soup.title.string else ""
            raw_text    = _extract_main_content(soup)
            description = re.sub(r"\s+", " ", raw_text).strip()
            if any(sig in description.lower() for sig in _BOT_SIGNALS):
                return "", fallback_snippet, False
            if len(description) < 150:
                return page_title, fallback_snippet, False
            return page_title, description[:8000], True
        except Exception as exc:
            log.debug("Scrape error [%s]: %s", url[:60], str(exc)[:60])
            break
    return "", fallback_snippet, False

def process_ats_result(res: dict, canonical_role: str) -> dict | None:
    url     = (res.get("href") or "").strip()
    snippet = (res.get("body") or "")
    s_title = (res.get("title") or "")
    if not url or is_garbage_url(url):
        return None
    if not url.startswith("http"):
        url = "https://" + url
    page_title, description, was_scraped = scrape_ats_url(url, snippet)
    if len(description.strip()) < 200:
        description = snippet
    raw_title = page_title if page_title else s_title
    return build_record(
        url=url, description=description, raw_title=raw_title,
        canonical_role=canonical_role, source="ats_ddgs", was_scraped=was_scraped,
    )

def _ddgs_search(query: str) -> list:
    for backend in GOOD_BACKENDS:
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(
                    query, backend=backend, region="us-en",
                    max_results=CONFIG["ddgs_max_results"],
                    timelimit=CONFIG["ddgs_time_filter"],
                ))
            if results:
                return results
        except Exception as exc:
            err = str(exc)
            if "403" in err or "atelimit" in err.lower():
                time.sleep(random.uniform(*CONFIG["ratelimit_sleep"]))
            elif "No results" not in err:
                log.debug("  backend=%s error: %s", backend, err[:80])
    return []

def fetch_ats_jobs(canonical_role: str, phrases: list) -> list:
    """DDGS site: search only on portals that don't block bots."""
    records:   list = []
    seen_urls: set  = set()

    for portal in ATS_PORTAL_SITES:
        raw_results = []
        for phrase in phrases:
            query = f'{portal} {phrase} job USA'
            log.info("  🔍 ATS: %s", query)
            raw_results = _ddgs_search(query)
            if raw_results:
                break
            time.sleep(random.uniform(2, 3))

        if not raw_results:
            time.sleep(random.uniform(*CONFIG["search_delay"]))
            continue

        log.info("    📥 %d ATS raw results", len(raw_results))

        new_results = [r for r in raw_results if r.get("href","") not in seen_urls]
        with ThreadPoolExecutor(max_workers=CONFIG["scrape_workers"]) as pool:
            futures = {pool.submit(process_ats_result, res, canonical_role): res
                       for res in new_results}
            try:
                for future in as_completed(futures, timeout=90):
                    try:
                        job = future.result()
                        if job and job["apply_link"] not in seen_urls:
                            seen_urls.add(job["apply_link"])
                            records.append(job)
                    except Exception as exc:
                        log.debug("Future error: %s", str(exc)[:60])
            except FutureTimeout:
                log.warning("  ⚠️  Some futures timed out")

        for r in raw_results:
            seen_urls.add(r.get("href",""))
        time.sleep(random.uniform(*CONFIG["search_delay"]))

    log.info("  ✅ ATS → %d valid jobs for '%s'", len(records), canonical_role)
    return records

# ============================================================
# SECTION 7: LAYER 3 — DICE RSS FEED (free, no auth)
# ============================================================

def fetch_dice_rss_jobs(role_label: str, role_slug: str) -> list:
    """
    Dice publishes a public RSS feed per search. No API key needed.
    Date filter r1 = posted in last 1 day.
    """
    url = (
        f"https://www.dice.com/jobs/q-{role_slug}-jobs.rss"
        f"?dateRange={CONFIG['dice_date_filter']}&country=US&language=en"
    )
    records = []
    try:
        feed = feedparser.parse(url)
        log.info("  📡 Dice RSS → %d entries for '%s'", len(feed.entries), role_slug)
        for entry in feed.entries:
            link  = entry.get("link", "")
            title = entry.get("title", "")
            desc  = entry.get("summary", "")
            if not link:
                continue
            rec = build_record(
                url=link, description=desc, raw_title=title,
                canonical_role=role_label, source="dice_rss", was_scraped=False,
            )
            if rec:
                records.append(rec)
    except Exception as exc:
        log.warning("  Dice RSS error: %s", str(exc)[:80])

    log.info("  ✅ Dice RSS → %d valid jobs for '%s'", len(records), role_label)
    return records

# ============================================================
# SECTION 8: DELTA LAKE WRITER
# ============================================================

JOB_SCHEMA = StructType([
    StructField("id",                  StringType(),    True),
    StructField("job_hash",            StringType(),    True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("fetch_date",          DateType(),      True),
    StructField("search_keyword",      StringType(),    True),
    StructField("source",              StringType(),    True),
    StructField("company_name",        StringType(),    True),
    StructField("job_title",           StringType(),    True),
    StructField("job_description",     StringType(),    True),
    StructField("apply_link",          StringType(),    True),
    StructField("hr_email",            StringType(),    True),
    StructField("job_type",            StringType(),    True),
    StructField("salary_range",        StringType(),    True),
    StructField("experience_required", StringType(),    True),
    StructField("location",            StringType(),    True),
    StructField("skills_required",     StringType(),    True),
    StructField("link_status",         StringType(),    True),
    StructField("validation_score",    IntegerType(),   True),
    StructField("validation_status",   StringType(),    True),
])

def ensure_table(spark: SparkSession, table: str):
    parts = table.split(".")
    if len(parts) == 3:
        catalog, schema, _ = parts
        for ddl in [
            f"CREATE CATALOG IF NOT EXISTS `{catalog}`",
            f"CREATE SCHEMA  IF NOT EXISTS `{catalog}`.`{schema}`",
        ]:
            try: spark.sql(ddl)
            except Exception: pass

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id                  STRING,
            job_hash            STRING,
            created_at          TIMESTAMP,
            updated_at          TIMESTAMP,
            fetch_date          DATE,
            search_keyword      STRING,
            source              STRING,
            company_name        STRING,
            job_title           STRING,
            job_description     STRING,
            apply_link          STRING,
            hr_email            STRING,
            job_type            STRING,
            salary_range        STRING,
            experience_required STRING,
            location            STRING,
            skills_required     STRING,
            link_status         STRING,
            validation_score    INT,
            validation_status   STRING
        )
        USING DELTA
        PARTITIONED BY (fetch_date)
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    # Add 'source' column if upgrading from V7
    try:
        existing = {c.lower() for c in spark.table(table).columns}
        if "source" not in existing:
            spark.sql(f"ALTER TABLE {table} ADD COLUMN source STRING")
            log.info("🔧 Added column: source STRING")
    except Exception as exc:
        log.warning("Schema migration: %s", exc)

def write_to_delta(records: list, spark: SparkSession) -> int:
    if not records:
        log.warning("No records to write.")
        return 0

    table = CONFIG["delta_table"]
    ensure_table(spark, table)

    df    = spark.createDataFrame(records, schema=JOB_SCHEMA)
    df    = df.dropDuplicates(["job_hash"])
    count = df.count()

    (
        DeltaTable.forName(spark, table).alias("tgt")
        .merge(df.alias("src"), "tgt.job_hash = src.job_hash")
        .whenMatchedUpdate(set={
            "updated_at":          "src.updated_at",
            "link_status":         "src.link_status",
            "hr_email":            "src.hr_email",
            "job_description":     "src.job_description",
            "job_type":            "src.job_type",
            "salary_range":        "src.salary_range",
            "skills_required":     "src.skills_required",
            "experience_required": "src.experience_required",
            "location":            "src.location",
            "validation_score":    "src.validation_score",
            "validation_status":   "src.validation_status",
            "source":              "src.source",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    return count

# ============================================================
# SECTION 9: MAIN ORCHESTRATOR
# ============================================================

def run_harvester():
    log.info("=" * 65)
    log.info("🚀 US IT JOB HARVESTER V8 — JSEARCH + ATS + DICE RSS")
    log.info("📅 Run date      : %s", date.today())
    log.info("🎯 Role groups   : %d", len(ROLES))
    log.info("🌐 LAYER 1       : JSearch API (LinkedIn/Indeed/Glassdoor/Dice/ZipRecruiter/Wellfound)")
    log.info("🌐 LAYER 2       : ATS direct scrape (Greenhouse/Lever/Ashby/Workable)")
    log.info("🌐 LAYER 3       : Dice RSS feed (free, no auth)")
    log.info("🔍 DDGS backend  : duckduckgo→startpage→brave (mojeek NEVER)")
    log.info("=" * 65)

    all_records: list = []

    for canonical_role, phrases in ROLES.items():
        log.info("\n📌 Role: %s", canonical_role)

        # LAYER 1: JSearch API — main LinkedIn/Indeed/Dice source
        # Use first phrase (most specific) for JSearch
        main_phrase = phrases[0].replace('"', '')
        jsearch_jobs = fetch_jsearch_jobs(canonical_role, main_phrase)
        all_records.extend(jsearch_jobs)

        # LAYER 2: ATS direct scraping (Greenhouse, Lever, etc.)
        ats_jobs = fetch_ats_jobs(canonical_role, phrases)
        all_records.extend(ats_jobs)

        # LAYER 3: Dice RSS (only for mapped roles)
        dice_slug = DICE_RSS_ROLES.get(canonical_role)
        if dice_slug:
            dice_jobs = fetch_dice_rss_jobs(canonical_role, dice_slug)
            all_records.extend(dice_jobs)

        time.sleep(random.uniform(5.0, 9.0))

    # Global dedup
    seen:   set  = set()
    unique: list = []
    for rec in all_records:
        h = rec["job_hash"]
        if h not in seen:
            seen.add(h)
            unique.append(rec)

    log.info("\n📊 Total collected       : %d", len(all_records))
    log.info("📊 After global dedup    : %d", len(unique))
    log.info("📊 From JSearch API      : %d", sum(1 for r in unique if r.get("source") == "jsearch"))
    log.info("📊 From ATS scrape       : %d", sum(1 for r in unique if r.get("source") == "ats_ddgs"))
    log.info("📊 From Dice RSS         : %d", sum(1 for r in unique if r.get("source") == "dice_rss"))
    log.info("📊 Valid   (score ≥ 70)  : %d", sum(1 for r in unique if r["validation_score"] >= 70))
    log.info("📊 Partial (score 45-69) : %d", sum(1 for r in unique if 45 <= r["validation_score"] < 70))

    written = write_to_delta(unique, spark)

    log.info("\n" + "=" * 65)
    log.info("🎉 HARVEST COMPLETE — %d records written to Delta", written)
    log.info("=" * 65)

    # Summary table
    spark.sql(f"""
        SELECT
            source,
            search_keyword,
            validation_status,
            COUNT(*)                                               AS jobs,
            ROUND(AVG(validation_score))                          AS avg_score,
            COUNT(CASE WHEN hr_email    != 'Not Found'     THEN 1 END) AS with_email,
            COUNT(CASE WHEN salary_range!= 'Not Specified' THEN 1 END) AS with_salary,
            COUNT(CASE WHEN job_type   != 'Not Specified'  THEN 1 END) AS with_type
        FROM {CONFIG['delta_table']}
        WHERE fetch_date = current_date()
        GROUP BY source, search_keyword, validation_status
        ORDER BY source, search_keyword, avg_score DESC
    """).show(100, truncate=False)

# ▶️  RUN
run_harvester()

In [0]:
%sql
select * from main.jobs_automation.raw_jobs_staging

In [0]:
# ============================================================
# US IT JOB HARVESTER V8 — JSEARCH API + ATS DIRECT SCRAPE
# ============================================================
# ROOT CAUSE FIX (Why V7 returned garbage links):
#   ❌ DuckDuckGo site:linkedin.com → LinkedIn blocks bots
#      → DDGS returns random tutorial pages (gardenerspath, pypi, etc.)
#   ❌ site:indeed.com, site:dice.com → same problem
#   ❌ 237-char OR queries confuse all search engines
#
# V8 ARCHITECTURE:
#   ✅ LAYER 1 — JSearch API (RapidAPI): LinkedIn + Indeed + Glassdoor
#               + Dice + ZipRecruiter + Wellfound — ONE API, all portals
#   ✅ LAYER 2 — Direct ATS scraping: Greenhouse + Lever + Ashby + Workable
#               (these have no bot protection, DDGS site: works fine)
#   ✅ LAYER 3 — Dice RSS feed (free, no auth needed)
#   ✅ Content validation score 0-100 (same as V7)
#   ✅ Delta MERGE dedup on job_hash
#   ✅ All jobs written to same Delta table as before
#
# ONE-TIME SETUP:
#   %pip install curl_cffi duckduckgo-search feedparser
# ============================================================

import hashlib
import uuid
import re
import time
import random
import feedparser
from bs4 import BeautifulSoup
from datetime import date, datetime
from duckduckgo_search import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DateType, TimestampType, IntegerType,
)
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutureTimeout
import warnings
import logging

try:
    from curl_cffi import requests as cf_requests
    _HAS_CURL_CFFI = True
except ImportError:
    import requests as cf_requests
    _HAS_CURL_CFFI = False

import requests as std_requests   # always available for JSearch API calls

warnings.filterwarnings("ignore", category=ResourceWarning)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ============================================================
# ⚙️  CONFIG
# ============================================================
CONFIG = {
    # Delta table (Unity Catalog)
    "delta_table": "main.jobs_automation.raw_jobs_staging",

    # JSearch API key — stored in Databricks secrets
    # Setup: dbutils.secrets.put("jobs_automation", "jsearch", "<your_key>")
    "jsearch_secret_scope": "jobs_automation",
    "jsearch_secret_key":   "jsearch",

    # JSearch pages per keyword (10 results/page → pages=5 → 50 jobs/keyword)
    "jsearch_pages": 5,

    # ATS scraping via DDGS (Greenhouse, Lever, Ashby, Workable)
    "ddgs_max_results": 15,
    "ddgs_time_filter": "d",          # 'd'=last day, 'w'=last week

    # Dice RSS date filter: 'r1'=last day, 'r7'=last week
    "dice_date_filter": "r1",

    # Concurrent scrape workers for ATS pages
    "scrape_workers": 4,

    # HTTP timeout
    "http_timeout": 12,

    # Minimum validation score to keep (0-100)
    "min_score": 45,

    # Delay between DDGS queries
    "search_delay": (5.0, 8.0),

    # Sleep on rate-limit
    "ratelimit_sleep": (15.0, 25.0),
}

# ============================================================
# ROLES — search terms for each layer
# ============================================================
ROLES = {
    "Data Engineer":             ['"Data Engineer"', '"Senior Data Engineer"'],
    "Spark / PySpark Engineer":  ['"PySpark Engineer"', '"Spark Engineer"'],
    "ETL Developer":             ['"ETL Developer"', '"ETL Engineer"'],
    "Analytics Engineer":        ['"Analytics Engineer"', '"BI Engineer"'],
    "Data Platform Engineer":    ['"Data Platform Engineer"'],
    "Cloud Data Engineer":       ['"Cloud Data Engineer"', '"AWS Data Engineer"'],
    "Databricks Engineer":       ['"Databricks Engineer"', '"Databricks Developer"'],
    "Python Developer":          ['"Python Developer"', '"Python Engineer"'],
    "Machine Learning Engineer": ['"Machine Learning Engineer"', '"MLOps Engineer"'],
}

# ============================================================
# ATS PORTALS — where DDGS site: search works reliably
# LinkedIn/Indeed/Dice NOT here — use JSearch API instead
# ============================================================
ATS_PORTAL_SITES = [
    "site:boards.greenhouse.io",
    "site:jobs.lever.co",
    "site:jobs.ashbyhq.com",
    "site:apply.workable.com",
    "site:myworkdayjobs.com",
    "site:smartrecruiters.com/jobs",
    "site:weworkremotely.com/remote-jobs",
    "site:remoteok.com",
]

# Dice RSS feeds (role slug → search URL)
DICE_RSS_ROLES = {
    "Data Engineer":             "data+engineer",
    "Python Developer":          "python+developer",
    "ETL Developer":             "etl+developer",
    "Machine Learning Engineer": "machine+learning+engineer",
    "Analytics Engineer":        "analytics+engineer",
    "Databricks Engineer":       "databricks+engineer",
}

GOOD_BACKENDS = ["duckduckgo", "startpage", "brave"]

SNIPPET_ONLY_DOMAINS: set = {
    "linkedin.com", "indeed.com", "dice.com", "glassdoor.com",
    "ziprecruiter.com", "monster.com", "wellfound.com",
    "careerbuilder.com",
}

GARBAGE_DOMAINS: set = {
    "wikipedia.org", "w3schools.com", "python.org", "geeksforgeeks.org",
    "tutorialspoint.com", "ibm.com/docs", "ibm.com/think",
    "learn.microsoft.com", "docs.databricks.com", "spark.apache.org",
    "youtube.com", "stackoverflow.com", "medium.com", "dev.to",
    "reddit.com", "quora.com", "coursera.org", "udemy.com",
    "data.gov", "cdc.gov", "va.gov", "nih.gov", "census.gov",
    "espn.com", "mathsisfun.com", "mygreatlearning.com",
    "gardenerspath.com", "pypi.org", "oracle.com/integration",
    "informatica.com/resources", "sas.com/insights",
    "ml-ops.org", "mechlesson.com", "kodekloud.com",
    "theengineeringchoice.org", "community.databricks.com",
    "bing.com", "google.com", "aws.amazon.com/products",
    "drive4spark", "sparkdriverapp",
}

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/130.0.0.0 Safari/537.36",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/131.0.0.0 Safari/537.36",
]

# ============================================================
# SECTION 1: URL FILTERS
# ============================================================

def is_garbage_url(url: str) -> bool:
    low = url.lower()
    return any(d in low for d in GARBAGE_DOMAINS)

def is_snippet_only(url: str) -> bool:
    low = url.lower()
    return any(d in low for d in SNIPPET_ONLY_DOMAINS)

# ============================================================
# SECTION 2: CONTENT VALIDATORS
# ============================================================

_JOB_KEYWORDS = [
    "engineer", "developer", "position", "experience", "skills",
    "requirements", "responsibilities", "hiring", "apply",
    "qualifications", "remote", "full-time", "salary", "benefits",
    "team", "role", "candidate", "opportunity", "employment",
]
_SPANISH_SIGNALS = ["años", "quinceanera", "quinceañera", "fiesta", "viaje", "celebra"]
_GERMAN_SIGNALS  = ["gmbh", "gründen", "buchhaltung", "kostenlos"]

def is_real_job_page(text: str) -> bool:
    if not text or len(text) < 120:
        return False
    low = text.lower()
    if sum(1 for s in _SPANISH_SIGNALS if s in low) >= 2:
        return False
    if sum(1 for s in _GERMAN_SIGNALS if s in low) >= 2:
        return False
    return sum(1 for kw in _JOB_KEYWORDS if kw in low) >= 3

def compute_validation_score(rec: dict) -> int:
    score = 0
    desc  = rec.get("job_description", "")
    dlen  = len(desc)

    if is_real_job_page(desc):           score += 25
    if dlen > 1500:                      score += 20
    elif dlen > 500:                     score += 12
    elif dlen > 200:                     score += 5

    if rec.get("company_name", "Unknown") not in ("Unknown", "", "Not Specified"):
        score += 15
    if rec.get("job_type", "Not Specified") != "Not Specified":
        score += 10
    if rec.get("skills_required", "Not Specified") != "Not Specified":
        score += 10
    if rec.get("salary_range", "Not Specified") != "Not Specified":
        score += 10
    if rec.get("hr_email", "Not Found") != "Not Found":
        score += 5
    if rec.get("experience_required", "Not Specified") != "Not Specified":
        score += 5
    return min(score, 100)

def get_validation_status(score: int) -> str:
    if score >= 70:   return "Valid"
    if score >= CONFIG["min_score"]: return "Partial"
    return "Junk"

# ============================================================
# SECTION 3: TEXT CLEANERS & EXTRACTORS
# ============================================================

_SPAM_SUFFIXES = [
    r" - LinkedIn", r" \| LinkedIn", r" - Greenhouse", r" - Lever",
    r" \| Wellfound", r" - Dice\.com", r" \| Built In", r" - Indeed",
    r" - ZipRecruiter", r" \| Glassdoor", r" - Monster",
    r" - Remote OK", r" - Jobvite", r" \| Workable",
    r" - SmartRecruiters", r" - Ashby",
]

def _clean_raw_title(raw: str) -> str:
    s = re.sub(r"\[.*?\]|\(.*?\)", "", raw)
    s = re.sub(r"(?i)(Job Application for|Apply for|Jobs In)\s+", "", s)
    for sfx in _SPAM_SUFFIXES:
        s = re.sub(rf"(?i){sfx}.*$", "", s)
    return s.strip()

def split_title_and_company(raw_title: str):
    cleaned   = _clean_raw_title(raw_title)
    job_title = cleaned
    company   = "Unknown"
    for sep in [" at ", " At ", " - ", " | ", " @ ", " – "]:
        if sep in cleaned:
            parts     = cleaned.split(sep, 1)
            job_title = parts[0].strip()
            company   = parts[1].strip()
            break
    job_title = re.sub(r"[^a-zA-Z0-9\s\+\#\./,]", "", job_title).strip().title()
    company   = re.sub(r"[^a-zA-Z0-9\s\.,&\-]",   "", company  ).strip().title()
    return (job_title or "Unknown"), (company or "Unknown")

def extract_emails(text: str) -> str:
    found = re.findall(r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,7}\b", text)
    _noise = {"noreply","no-reply","donotreply","notifications","mailer","bounce",
              "alert","privacy","legal","unsubscribe","sentry","example.com",
              "support","help","feedback","abuse","postmaster"}
    clean = [e for e in found if not any(n in e.lower() for n in _noise) and len(e) > 6]
    return ", ".join(sorted(set(clean))) if clean else "Not Found"

def detect_job_type(desc: str) -> str:
    d = desc.lower(); tags = []
    if re.search(r"\bc2c\b|corp[\s\-]?to[\s\-]?corp", d):           tags.append("C2C")
    if re.search(r"\bw[\s\-]?2\b", d):                               tags.append("W2")
    if re.search(r"\b1099\b", d):                                    tags.append("1099")
    if re.search(r"\bcontract[\s\-]to[\s\-]hire\b|\bc2h\b", d):     tags.append("Contract-to-Hire")
    elif re.search(r"\bcontract\b|\bcontractor\b", d):               tags.append("Contract")
    if re.search(r"\bfull[\s\-]?time\b|\bpermanent\b|\bfte\b", d):  tags.append("Full-Time")
    if re.search(r"\bpart[\s\-]?time\b", d):                        tags.append("Part-Time")
    if re.search(r"\bremote\b|work[\s\-]from[\s\-]home|\bwfh\b", d): tags.append("Remote")
    if re.search(r"\bhybrid\b", d):                                  tags.append("Hybrid")
    if re.search(r"\bon[\s\-]?site\b|in[\s\-]?office\b", d):        tags.append("Onsite")
    return ", ".join(tags) if tags else "Not Specified"

def extract_salary(desc: str) -> str:
    patterns = [
        r"\$[\d,]+(?:\.\d{2})?\s*[-–to]+\s*\$[\d,]+(?:\.\d{2})?(?:\s*(?:k|K|/hr|/hour|/year|/yr|annually))?",
        r"\$[\d,]+(?:\.\d{2})?(?:\s*(?:k|K|/hr|/hour|/year|/yr|annually))",
        r"[\d,]+\s*[-–]\s*[\d,]+\s*(?:USD|k|K)\s*(?:per\s+(?:year|hr|hour)|annually|/hr)",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m: return m.group(0).strip()
    return "Not Specified"

def extract_experience(desc: str) -> str:
    patterns = [
        r"\d+\+?\s*[-–to]+\s*\d+\s+years?\s+(?:of\s+)?(?:professional\s+)?experience",
        r"\d+\+?\s+years?\s+(?:of\s+)?(?:professional\s+)?experience",
        r"(?:minimum|min\.?|at\s+least)\s+\d+\+?\s+years?",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m: return m.group(0).strip()
    return "Not Specified"

def extract_location(desc: str) -> str:
    _states = ("AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|"
               "MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|"
               "TX|UT|VT|VA|WA|WV|WI|WY|DC")
    patterns = [
        rf"(?:location|located\s+in|based\s+in|office\s+in)\s*[:\-]?\s*([A-Za-z\s]+,?\s*(?:{_states}|USA|United\s+States))",
        rf"([A-Za-z][A-Za-z\s]+,\s*(?:{_states}))\b",
        r"(?:Remote|Hybrid|Onsite)[,\s]+(?:in\s+)?([A-Za-z\s,]{4,40})",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m:
            loc = m.group(1).strip().rstrip(",").strip()
            if 2 < len(loc) < 60: return loc
    return "Not Specified"

_SKILLS = [
    (r"\bPython\b","Python"),(r"\bScala\b","Scala"),(r"\bJava\b","Java"),
    (r"\b(?:SQL|T-SQL|PL/SQL)\b","SQL"),(r"\bPySpark\b","PySpark"),
    (r"\b(?:Apache\s+)?Spark\b","Spark"),(r"\b(?:Apache\s+)?Kafka\b","Kafka"),
    (r"\b(?:Apache\s+)?Flink\b","Flink"),(r"\b(?:Apache\s+)?Hadoop\b","Hadoop"),
    (r"\bHive\b","Hive"),(r"\bKinesis\b","Kinesis"),
    (r"\bAWS\b|Amazon\s+Web\s+Services","AWS"),(r"\bAzure\b|Microsoft\s+Azure","Azure"),
    (r"\bGCP\b|Google\s+Cloud","GCP"),(r"\bDatabricks\b","Databricks"),
    (r"\bSnowflake\b","Snowflake"),(r"\bRedshift\b","Redshift"),
    (r"\bBigQuery\b","BigQuery"),(r"\bDelta\s+Lake\b","Delta Lake"),
    (r"\bIceberg\b","Iceberg"),(r"\bAirflow\b","Airflow"),
    (r"\bdbt\b|data\s+build\s+tool","dbt"),(r"\bTerraform\b","Terraform"),
    (r"\bDocker\b","Docker"),(r"\bKubernetes\b|\bK8s\b","Kubernetes"),
    (r"\bPostgreSQL\b|\bpostgres\b","PostgreSQL"),(r"\bMySQL\b","MySQL"),
    (r"\bMongoDB\b","MongoDB"),(r"\bMLflow\b","MLflow"),
    (r"\bPower\s+BI\b","Power BI"),(r"\bTableau\b","Tableau"),(r"\bLooker\b","Looker"),
]

def extract_skills(desc: str) -> str:
    found = []
    for pattern, label in _SKILLS:
        if re.search(pattern, desc, re.IGNORECASE) and label not in found:
            found.append(label)
    return ", ".join(found[:30]) if found else "Not Specified"

# ============================================================
# SECTION 4: RECORD BUILDER (shared by all layers)
# ============================================================

_EXPIRED_SIGNALS = [
    "job no longer available", "position has been filled",
    "this job has expired", "no longer accepting applications",
    "job has been closed", "listing has expired",
]

def build_record(
    url: str,
    description: str,
    raw_title: str,
    canonical_role: str,
    source: str = "unknown",
    was_scraped: bool = True,
) -> dict | None:
    """Build + validate one job record. Returns None if below threshold."""
    if not url or is_garbage_url(url):
        return None
    if not is_real_job_page(description):
        log.debug("⛔ Not a job page: %s", url[:70])
        return None

    low = description.lower()
    if any(s in low for s in _EXPIRED_SIGNALS):
        log.info("⏩ Expired: %s", url[:70])
        return None

    job_title, company = split_title_and_company(raw_title)
    if not job_title or len(job_title.strip()) < 2:
        job_title = canonical_role.title()

    link_status = "Active" if was_scraped else "Unverified"
    now = datetime.now()

    rec = {
        "id":                  str(uuid.uuid4()),
        "job_hash":            hashlib.md5(url.encode()).hexdigest(),
        "created_at":          now,
        "updated_at":          now,
        "fetch_date":          now.date(),
        "search_keyword":      canonical_role,
        "source":              source,
        "company_name":        company,
        "job_title":           job_title,
        "job_description":     description,
        "apply_link":          url,
        "hr_email":            extract_emails(description),
        "job_type":            detect_job_type(description),
        "salary_range":        extract_salary(description),
        "experience_required": extract_experience(description),
        "location":            extract_location(description),
        "skills_required":     extract_skills(description),
        "link_status":         link_status,
        "validation_score":    0,
        "validation_status":   "Pending",
    }
    score                   = compute_validation_score(rec)
    rec["validation_score"] = score
    rec["validation_status"]= get_validation_status(score)

    if score < CONFIG["min_score"]:
        log.debug("⛔ Score %d < %d: %s", score, CONFIG["min_score"], url[:70])
        return None
    return rec

# ============================================================
# SECTION 5: LAYER 1 — JSEARCH API
# Covers: LinkedIn, Indeed, Glassdoor, Dice, ZipRecruiter, Wellfound
# ============================================================

def _get_jsearch_key() -> str:
    try:
        return dbutils.secrets.get(
            CONFIG["jsearch_secret_scope"],
            CONFIG["jsearch_secret_key"],
        )
    except Exception:
        raise RuntimeError(
            "❌ JSearch API key not found in Databricks secrets.\n"
            "   Run: dbutils.secrets.put('jobs_automation', 'jsearch', '<your_key>')"
        )

def fetch_jsearch_jobs(role_label: str, search_phrase: str) -> list:
    """
    Call JSearch API for one role. Returns list of validated records.
    JSearch covers: LinkedIn, Indeed, Glassdoor, Dice, ZipRecruiter, Wellfound.
    """
    try:
        api_key = _get_jsearch_key()
    except RuntimeError as e:
        log.error(str(e))
        return []

    url     = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key":  api_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com",
    }

    all_jobs: list = []

    for page in range(1, CONFIG["jsearch_pages"] + 1):
        params = {
            "query":          f"{search_phrase} USA",
            "page":           str(page),
            "num_pages":      "1",
            "date_posted":    "today",   # only today's jobs!
            "employment_types": "FULLTIME,CONTRACTOR,PARTTIME",
        }
        try:
            resp = std_requests.get(url, headers=headers, params=params, timeout=20)
            if resp.status_code == 429:
                log.warning("  ⚠️  JSearch rate limit — sleeping 30s")
                time.sleep(30)
                continue
            if resp.status_code != 200:
                log.warning("  JSearch HTTP %d for page %d", resp.status_code, page)
                break

            data     = resp.json()
            jobs_raw = data.get("data", [])
            if not jobs_raw:
                break   # no more pages

            log.info("  📥 JSearch page %d → %d jobs for '%s'", page, len(jobs_raw), search_phrase)

            for job in jobs_raw:
                if not isinstance(job, dict):
                    continue

                desc   = job.get("job_description", "") or ""
                title  = job.get("job_title", "")        or ""
                company= job.get("employer_name", "")    or "Unknown"
                link   = (
                    job.get("job_apply_link")
                    or job.get("job_google_link")
                    or ""
                )
                if not link:
                    continue

                raw_title = f"{title} at {company}" if company and company != "Unknown" else title

                # Enrich description with structured fields JSearch provides
                extra = []
                if job.get("job_city"):       extra.append(f"Location: {job['job_city']}, {job.get('job_state','')}")
                if job.get("job_min_salary"): extra.append(f"Salary: ${job['job_min_salary']:,} - ${job.get('job_max_salary', job['job_min_salary']):,} {job.get('job_salary_period','')}")
                if job.get("job_employment_type"): extra.append(f"Type: {job['job_employment_type']}")
                if job.get("job_required_skills"):  extra.append(f"Skills: {', '.join(job['job_required_skills'])}")
                if extra:
                    desc = "\n".join(extra) + "\n\n" + desc

                rec = build_record(
                    url=link,
                    description=desc[:8000],
                    raw_title=raw_title,
                    canonical_role=role_label,
                    source="jsearch",
                    was_scraped=True,
                )
                if rec:
                    all_jobs.append(rec)

            time.sleep(random.uniform(1.0, 2.0))   # polite gap between pages

        except Exception as exc:
            log.error("  JSearch error page %d: %s", page, str(exc)[:80])
            break

    log.info("  ✅ JSearch → %d valid jobs for '%s'", len(all_jobs), role_label)
    return all_jobs

# ============================================================
# SECTION 6: LAYER 2 — DIRECT ATS SCRAPING (Greenhouse/Lever/etc.)
# Only for portals where DDGS site: search actually works
# ============================================================

def _make_session():
    if _HAS_CURL_CFFI:
        return cf_requests.Session(impersonate="chrome131")
    import requests as _r
    s = _r.Session()
    s.headers.update({"User-Agent": random.choice(USER_AGENTS)})
    return s

_SCRAPE_SESSION = _make_session()

_CONTENT_IDS      = ["content","job-description","jobDescription","job_description","main-content"]
_CONTENT_CLASSES  = [re.compile(p, re.IGNORECASE) for p in [
    r"job[\-_]?description", r"job[\-_]?details", r"\bdescription\b", r"\bcontent\b",
]]
_BOT_SIGNALS      = ["checking your browser","access denied","cloudflare","captcha","enable javascript","just a moment"]

def _extract_main_content(soup: BeautifulSoup) -> str:
    for id_val in _CONTENT_IDS:
        el = soup.find(id=id_val)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    for pat in _CONTENT_CLASSES:
        el = soup.find(class_=pat)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    for tag in ["main","article"]:
        el = soup.find(tag)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    return soup.get_text(separator=" ", strip=True)

def scrape_ats_url(url: str, fallback_snippet: str = "") -> tuple:
    """Returns (page_title, description, was_scraped)."""
    if is_snippet_only(url):
        return "", fallback_snippet, False
    for _ in range(2):
        try:
            resp = _SCRAPE_SESSION.get(url, timeout=CONFIG["http_timeout"], allow_redirects=True)
            if resp.status_code in (403, 429, 503):
                time.sleep(random.uniform(4, 8))
                continue
            if resp.status_code != 200:
                return "", fallback_snippet, False
            soup = BeautifulSoup(resp.text, "lxml")
            for tag in soup(["script","style","footer","nav","header","aside","noscript","iframe"]):
                tag.decompose()
            page_title  = soup.title.string.strip() if soup.title and soup.title.string else ""
            raw_text    = _extract_main_content(soup)
            description = re.sub(r"\s+", " ", raw_text).strip()
            if any(sig in description.lower() for sig in _BOT_SIGNALS):
                return "", fallback_snippet, False
            if len(description) < 150:
                return page_title, fallback_snippet, False
            return page_title, description[:8000], True
        except Exception as exc:
            log.debug("Scrape error [%s]: %s", url[:60], str(exc)[:60])
            break
    return "", fallback_snippet, False

def process_ats_result(res: dict, canonical_role: str) -> dict | None:
    url     = (res.get("href") or "").strip()
    snippet = (res.get("body") or "")
    s_title = (res.get("title") or "")
    if not url or is_garbage_url(url):
        return None
    if not url.startswith("http"):
        url = "https://" + url
    page_title, description, was_scraped = scrape_ats_url(url, snippet)
    if len(description.strip()) < 200:
        description = snippet
    raw_title = page_title if page_title else s_title
    return build_record(
        url=url, description=description, raw_title=raw_title,
        canonical_role=canonical_role, source="ats_ddgs", was_scraped=was_scraped,
    )

def _ddgs_search(query: str) -> list:
    for backend in GOOD_BACKENDS:
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(
                    query, backend=backend, region="us-en",
                    max_results=CONFIG["ddgs_max_results"],
                    timelimit=CONFIG["ddgs_time_filter"],
                ))
            if results:
                return results
        except Exception as exc:
            err = str(exc)
            if "403" in err or "atelimit" in err.lower():
                time.sleep(random.uniform(*CONFIG["ratelimit_sleep"]))
            elif "No results" not in err:
                log.debug("  backend=%s error: %s", backend, err[:80])
    return []

def fetch_ats_jobs(canonical_role: str, phrases: list) -> list:
    """DDGS site: search only on portals that don't block bots."""
    records:   list = []
    seen_urls: set  = set()

    for portal in ATS_PORTAL_SITES:
        raw_results = []
        for phrase in phrases:
            query = f'{portal} {phrase} job USA'
            log.info("  🔍 ATS: %s", query)
            raw_results = _ddgs_search(query)
            if raw_results:
                break
            time.sleep(random.uniform(2, 3))

        if not raw_results:
            time.sleep(random.uniform(*CONFIG["search_delay"]))
            continue

        log.info("    📥 %d ATS raw results", len(raw_results))

        new_results = [r for r in raw_results if r.get("href","") not in seen_urls]
        with ThreadPoolExecutor(max_workers=CONFIG["scrape_workers"]) as pool:
            futures = {pool.submit(process_ats_result, res, canonical_role): res
                       for res in new_results}
            try:
                for future in as_completed(futures, timeout=90):
                    try:
                        job = future.result()
                        if job and job["apply_link"] not in seen_urls:
                            seen_urls.add(job["apply_link"])
                            records.append(job)
                    except Exception as exc:
                        log.debug("Future error: %s", str(exc)[:60])
            except FutureTimeout:
                log.warning("  ⚠️  Some futures timed out")

        for r in raw_results:
            seen_urls.add(r.get("href",""))
        time.sleep(random.uniform(*CONFIG["search_delay"]))

    log.info("  ✅ ATS → %d valid jobs for '%s'", len(records), canonical_role)
    return records

# ============================================================
# SECTION 7: LAYER 3 — DICE RSS FEED (free, no auth)
# ============================================================

def fetch_dice_rss_jobs(role_label: str, role_slug: str) -> list:
    """
    Dice publishes a public RSS feed per search. No API key needed.
    Date filter r1 = posted in last 1 day.
    """
    url = (
        f"https://www.dice.com/jobs/q-{role_slug}-jobs.rss"
        f"?dateRange={CONFIG['dice_date_filter']}&country=US&language=en"
    )
    records = []
    try:
        feed = feedparser.parse(url)
        log.info("  📡 Dice RSS → %d entries for '%s'", len(feed.entries), role_slug)
        for entry in feed.entries:
            link  = entry.get("link", "")
            title = entry.get("title", "")
            desc  = entry.get("summary", "")
            if not link:
                continue
            rec = build_record(
                url=link, description=desc, raw_title=title,
                canonical_role=role_label, source="dice_rss", was_scraped=False,
            )
            if rec:
                records.append(rec)
    except Exception as exc:
        log.warning("  Dice RSS error: %s", str(exc)[:80])

    log.info("  ✅ Dice RSS → %d valid jobs for '%s'", len(records), role_label)
    return records

# ============================================================
# SECTION 8: DELTA LAKE WRITER
# ============================================================

JOB_SCHEMA = StructType([
    StructField("id",                  StringType(),    True),
    StructField("job_hash",            StringType(),    True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("fetch_date",          DateType(),      True),
    StructField("search_keyword",      StringType(),    True),
    StructField("source",              StringType(),    True),
    StructField("company_name",        StringType(),    True),
    StructField("job_title",           StringType(),    True),
    StructField("job_description",     StringType(),    True),
    StructField("apply_link",          StringType(),    True),
    StructField("hr_email",            StringType(),    True),
    StructField("job_type",            StringType(),    True),
    StructField("salary_range",        StringType(),    True),
    StructField("experience_required", StringType(),    True),
    StructField("location",            StringType(),    True),
    StructField("skills_required",     StringType(),    True),
    StructField("link_status",         StringType(),    True),
    StructField("validation_score",    IntegerType(),   True),
    StructField("validation_status",   StringType(),    True),
])

def ensure_table(spark: SparkSession, table: str):
    parts = table.split(".")
    if len(parts) == 3:
        catalog, schema, _ = parts
        for ddl in [
            f"CREATE CATALOG IF NOT EXISTS `{catalog}`",
            f"CREATE SCHEMA  IF NOT EXISTS `{catalog}`.`{schema}`",
        ]:
            try: spark.sql(ddl)
            except Exception: pass

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id                  STRING,
            job_hash            STRING,
            created_at          TIMESTAMP,
            updated_at          TIMESTAMP,
            fetch_date          DATE,
            search_keyword      STRING,
            source              STRING,
            company_name        STRING,
            job_title           STRING,
            job_description     STRING,
            apply_link          STRING,
            hr_email            STRING,
            job_type            STRING,
            salary_range        STRING,
            experience_required STRING,
            location            STRING,
            skills_required     STRING,
            link_status         STRING,
            validation_score    INT,
            validation_status   STRING
        )
        USING DELTA
        PARTITIONED BY (fetch_date)
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    # Add 'source' column if upgrading from V7
    try:
        existing = {c.lower() for c in spark.table(table).columns}
        if "source" not in existing:
            spark.sql(f"ALTER TABLE {table} ADD COLUMN source STRING")
            log.info("🔧 Added column: source STRING")
    except Exception as exc:
        log.warning("Schema migration: %s", exc)

def write_to_delta(records: list, spark: SparkSession) -> int:
    if not records:
        log.warning("No records to write.")
        return 0

    table = CONFIG["delta_table"]
    ensure_table(spark, table)

    df    = spark.createDataFrame(records, schema=JOB_SCHEMA)
    df    = df.dropDuplicates(["job_hash"])
    count = df.count()

    (
        DeltaTable.forName(spark, table).alias("tgt")
        .merge(df.alias("src"), "tgt.job_hash = src.job_hash")
        .whenMatchedUpdate(set={
            "updated_at":          "src.updated_at",
            "link_status":         "src.link_status",
            "hr_email":            "src.hr_email",
            "job_description":     "src.job_description",
            "job_type":            "src.job_type",
            "salary_range":        "src.salary_range",
            "skills_required":     "src.skills_required",
            "experience_required": "src.experience_required",
            "location":            "src.location",
            "validation_score":    "src.validation_score",
            "validation_status":   "src.validation_status",
            "source":              "src.source",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    return count

# ============================================================
# SECTION 9: MAIN ORCHESTRATOR
# ============================================================

def run_harvester():
    log.info("=" * 65)
    log.info("🚀 US IT JOB HARVESTER V8 — JSEARCH + ATS + DICE RSS")
    log.info("📅 Run date      : %s", date.today())
    log.info("🎯 Role groups   : %d", len(ROLES))
    log.info("🌐 LAYER 1       : JSearch API (LinkedIn/Indeed/Glassdoor/Dice/ZipRecruiter/Wellfound)")
    log.info("🌐 LAYER 2       : ATS direct scrape (Greenhouse/Lever/Ashby/Workable)")
    log.info("🌐 LAYER 3       : Dice RSS feed (free, no auth)")
    log.info("🔍 DDGS backend  : duckduckgo→startpage→brave (mojeek NEVER)")
    log.info("=" * 65)

    all_records: list = []

    for canonical_role, phrases in ROLES.items():
        log.info("\n📌 Role: %s", canonical_role)

        # LAYER 1: JSearch API — main LinkedIn/Indeed/Dice source
        # Use first phrase (most specific) for JSearch
        main_phrase = phrases[0].replace('"', '')
        jsearch_jobs = fetch_jsearch_jobs(canonical_role, main_phrase)
        all_records.extend(jsearch_jobs)

        # LAYER 2: ATS direct scraping (Greenhouse, Lever, etc.)
        ats_jobs = fetch_ats_jobs(canonical_role, phrases)
        all_records.extend(ats_jobs)

        # LAYER 3: Dice RSS (only for mapped roles)
        dice_slug = DICE_RSS_ROLES.get(canonical_role)
        if dice_slug:
            dice_jobs = fetch_dice_rss_jobs(canonical_role, dice_slug)
            all_records.extend(dice_jobs)

        time.sleep(random.uniform(5.0, 9.0))

    # Global dedup
    seen:   set  = set()
    unique: list = []
    for rec in all_records:
        h = rec["job_hash"]
        if h not in seen:
            seen.add(h)
            unique.append(rec)

    log.info("\n📊 Total collected       : %d", len(all_records))
    log.info("📊 After global dedup    : %d", len(unique))
    log.info("📊 From JSearch API      : %d", sum(1 for r in unique if r.get("source") == "jsearch"))
    log.info("📊 From ATS scrape       : %d", sum(1 for r in unique if r.get("source") == "ats_ddgs"))
    log.info("📊 From Dice RSS         : %d", sum(1 for r in unique if r.get("source") == "dice_rss"))
    log.info("📊 Valid   (score ≥ 70)  : %d", sum(1 for r in unique if r["validation_score"] >= 70))
    log.info("📊 Partial (score 45-69) : %d", sum(1 for r in unique if 45 <= r["validation_score"] < 70))

    written = write_to_delta(unique, spark)

    log.info("\n" + "=" * 65)
    log.info("🎉 HARVEST COMPLETE — %d records written to Delta", written)
    log.info("=" * 65)

    # Summary table
    spark.sql(f"""
        SELECT
            source,
            search_keyword,
            validation_status,
            COUNT(*)                                               AS jobs,
            ROUND(AVG(validation_score))                          AS avg_score,
            COUNT(CASE WHEN hr_email    != 'Not Found'     THEN 1 END) AS with_email,
            COUNT(CASE WHEN salary_range!= 'Not Specified' THEN 1 END) AS with_salary,
            COUNT(CASE WHEN job_type   != 'Not Specified'  THEN 1 END) AS with_type
        FROM {CONFIG['delta_table']}
        WHERE fetch_date = current_date()
        GROUP BY source, search_keyword, validation_status
        ORDER BY source, search_keyword, avg_score DESC
    """).show(100, truncate=False)

# ▶️  RUN
run_harvester()

In [0]:
# ============================================================
# US IT JOB HARVESTER V7 — BROWSER-GRADE + SMART SEARCH
# ============================================================
# FIXES vs V6:
#   🔴 ROOT CAUSE 1 FIXED: Force backend='duckduckgo' — never
#      use mojeek/grokipedia which ignore site: operator and
#      return garbage (Spanish websites, ESPN, IBM docs, etc.)
#   🔴 ROOT CAUSE 2 FIXED: Short focused queries (<60 chars)
#      instead of 237-char 15-location OR blobs
#   🔴 ROOT CAUSE 3 FIXED: 88 queries instead of 176 — fewer
#      portals, smart role aliases, backend rotation on 403
#   🔴 ROOT CAUSE 4 FIXED: curl_cffi impersonates Chrome —
#      bypasses 403 on Greenhouse, Lever, Workday, etc.
#   ✅ NEW: Content validation score 0-100
#   ✅ NEW: English + job-page detection (skips non-job content)
#   ✅ NEW: Role aliases (PySpark → Spark Engineer, etc.)
#   ✅ NEW: validation_score column in Delta table
# ============================================================
# ONE-TIME SETUP (run in Databricks before scheduling):
#   %pip install curl_cffi
# ============================================================

import hashlib
import uuid
import re
import time
import random
import feedparser
from bs4 import BeautifulSoup
from datetime import date, datetime
from duckduckgo_search import DDGS          # pip install duckduckgo-search
                                             # OR: from ddgs import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DateType, TimestampType, IntegerType,
)
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutureTimeout
import warnings
import logging

try:
    from curl_cffi import requests as cf_requests   # pip install curl_cffi
    _HAS_CURL_CFFI = True
except ImportError:
    import requests as cf_requests                  # fallback
    _HAS_CURL_CFFI = False

warnings.filterwarnings("ignore", category=ResourceWarning)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ============================================================
# ⚙️  CONFIG
# ============================================================
CONFIG = {
    # delta table (Unity Catalog: catalog.schema.table)
    "delta_table": "main.jobs_automation.raw_jobs_staging",

    # Results per DDGS query
    "max_results": 15,

    # Concurrent scrape workers
    "scrape_workers": 4,

    # DDGS time filter: 'd'=last day, 'w'=last week
    "time_filter": "d",

    # HTTP timeout (seconds)
    "http_timeout": 12,

    # Minimum validation score to keep a record (0-100)
    "min_score": 45,

    # Delay between DDGS queries (seconds) — avoids 403 cascade
    "search_delay": (6.0, 9.0),

    # Extra sleep when 403/rate-limit hit
    "ratelimit_sleep": (15.0, 25.0),
}

# ============================================================
# ROLE CONFIG — with aliases to maximise search hits
# ============================================================
# Map: canonical role → list of search phrases to try
# Order matters — first phrase attempted first; others are fallbacks
ROLES = {
    "Data Engineer": [
        '"Data Engineer"',
        '"Senior Data Engineer"',
    ],
    "Spark / PySpark Engineer": [
        '"Spark Engineer"',
        '"PySpark Engineer"',
        '"PySpark Developer"',
    ],
    "ETL Developer": [
        '"ETL Developer"',
        '"ETL Engineer"',
        '"Data Pipeline Engineer"',
    ],
    "Analytics Engineer": [
        '"Analytics Engineer"',
        '"BI Engineer"',
    ],
    "Data Platform Engineer": [
        '"Data Platform Engineer"',
        '"Data Infrastructure Engineer"',
    ],
    "Cloud Data Engineer": [
        '"Cloud Data Engineer"',
        '"AWS Data Engineer"',
        '"Azure Data Engineer"',
    ],
    "Databricks Engineer": [
        '"Databricks Engineer"',
        '"Databricks Developer"',
    ],
    "Python Developer": [
        '"Python Developer"',
        '"Python Engineer"',
    ],
    "Machine Learning Engineer": [
        '"Machine Learning Engineer"',
        '"MLOps Engineer"',
    ],
}

# ============================================================
# PORTALS (8 focused ATS + job sites)
# Reduced from 16 → 88 total queries instead of 176
# ============================================================
PORTAL_SITES = [
    "site:boards.greenhouse.io",
    "site:jobs.lever.co",
    "site:jobs.ashbyhq.com",
    "site:apply.workable.com",
    "site:myworkdayjobs.com",
    "site:smartrecruiters.com/jobs",
    "site:weworkremotely.com/remote-jobs",
    "site:remoteok.com",
]

# Backends that properly support the site: operator
# NEVER use mojeek / grokipedia / yahoo for site: queries
GOOD_BACKENDS = ["duckduckgo", "startpage", "brave"]

# Sites where we skip scraping and use snippet only
SNIPPET_ONLY: set = {
    "linkedin.com", "dice.com", "wellfound.com",
    "ziprecruiter.com", "monster.com", "indeed.com",
    "glassdoor.com", "careerbuilder.com",
}

# Garbage domains to reject outright
GARBAGE_DOMAINS: set = {
    # Tutorial / doc sites
    "wikipedia.org", "w3schools.com", "python.org", "geeksforgeeks.org",
    "tutorialspoint.com", "codecademy.com", "pluralsight.com",
    "ibm.com/docs", "ibm.com/think", "learn.microsoft.com",
    "docs.databricks.com", "spark.apache.org", "cloud.google.com/docs",
    # Non-job sites
    "youtube.com", "github.com/topics", "stackoverflow.com",
    "medium.com", "dev.to", "reddit.com", "quora.com",
    "coursera.org", "udemy.com", "hackerearth.com",
    "glassdoor.com/Interview", "glassdoor.com/Learn",
    # Common false-positive domains
    "data.gov", "cdc.gov", "va.gov", "nih.gov", "census.gov",
    "espn.com", "mathsisfun.com", "mygreatlearning.com",
    "comptia.org", "merriam-webster.com",
    # Sign-in / product pages (not job postings)
    "bing.com", "google.com", "aws.amazon.com",
    "azure.microsoft.com", "developers.google.com",
    "icloud.com", "onedrive.live.com",
    # Spanish / quinceañera sites (were appearing in V6 due to mojeek)
    "ef.com", "viajes", "quinceaera", "fifteens.com", "dreams15",
    "viajesnatours", "viajesjazmine", "raketenstart.de", "eltngl.com",
    # Drive/delivery apps falsely matched "Spark"
    "drive4spark", "sparkdriverapp",
}

# Rotating user-agents
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "Chrome/131.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "Chrome/130.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:133.0) Gecko/20100101 "
    "Firefox/133.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "Chrome/131.0.0.0 Safari/537.36",
]

# ============================================================
# SECTION 1: URL FILTERS
# ============================================================

def is_garbage_url(url: str) -> bool:
    low = url.lower()
    return any(d in low for d in GARBAGE_DOMAINS)

def is_snippet_only_url(url: str) -> bool:
    low = url.lower()
    return any(d in low for d in SNIPPET_ONLY)

# ============================================================
# SECTION 2: CONTENT VALIDATORS
# ============================================================

_JOB_KEYWORDS = [
    "engineer", "developer", "position", "experience", "skills",
    "requirements", "responsibilities", "hiring", "apply",
    "qualifications", "remote", "full-time", "salary", "benefits",
    "team", "role", "candidate", "opportunity", "employment",
]

_SPANISH_SIGNALS = [
    "años", "quinceanera", "quinceañera", "fiesta", "viaje",
    "celebra", "nuestro", "buscas", "comenzar", "empresa",
    "haz clic", "haga clic", "ingresa", "conoce",
]

_GERMAN_SIGNALS = [
    "gmbh", "gründen", "buchhaltung", "kostenlos", "digitalen",
]

def is_real_job_page(text: str) -> bool:
    """Return True only if content looks like an English job posting."""
    if not text or len(text) < 120:
        return False
    low = text.lower()

    # Reject non-English pages
    if sum(1 for s in _SPANISH_SIGNALS if s in low) >= 2:
        return False
    if sum(1 for s in _GERMAN_SIGNALS if s in low) >= 2:
        return False

    # Must have ≥3 job-related English terms
    return sum(1 for kw in _JOB_KEYWORDS if kw in low) >= 3

def compute_validation_score(rec: dict) -> int:
    """
    Score a job record 0-100 based on data completeness and quality.
    Threshold for keeping: CONFIG['min_score'] (default 45).
    """
    score = 0
    desc  = rec.get("job_description", "")
    low   = desc.lower()

    # 1. Real job page content (25 pts)
    if is_real_job_page(desc):
        score += 25

    # 2. Description length (20 pts)
    dlen = len(desc)
    if dlen > 1500:
        score += 20
    elif dlen > 500:
        score += 12
    elif dlen > 200:
        score += 5

    # 3. Real company name (15 pts)
    company = rec.get("company_name", "Unknown")
    if company and company not in ("Unknown", "", "Not Specified"):
        score += 15

    # 4. Job type detected (10 pts)
    if rec.get("job_type", "Not Specified") != "Not Specified":
        score += 10

    # 5. Skills detected (10 pts)
    if rec.get("skills_required", "Not Specified") != "Not Specified":
        score += 10

    # 6. Salary detected (10 pts)
    if rec.get("salary_range", "Not Specified") != "Not Specified":
        score += 10

    # 7. HR email found (5 pts)
    if rec.get("hr_email", "Not Found") != "Not Found":
        score += 5

    # 8. Experience requirement found (5 pts)
    if rec.get("experience_required", "Not Specified") != "Not Specified":
        score += 5

    return min(score, 100)

def get_validation_status(score: int) -> str:
    if score >= 70:
        return "Valid"
    if score >= CONFIG["min_score"]:
        return "Partial"
    return "Junk"

# ============================================================
# SECTION 3: TEXT CLEANERS
# ============================================================

_SPAM_SUFFIXES = [
    r" - LinkedIn", r" \| LinkedIn", r" - Greenhouse", r" - Lever",
    r" \| Wellfound", r" - Dice\.com", r" \| Built In", r" - Indeed",
    r" - ZipRecruiter", r" \| Glassdoor", r" - Monster",
    r" - Remote OK", r" - Jobvite", r" \| Workable",
    r" - SmartRecruiters", r" - Ashby",
]

def _clean_raw_title(raw: str) -> str:
    s = re.sub(r"\[.*?\]|\(.*?\)", "", raw)
    s = re.sub(r"(?i)(Job Application for|Apply for|Jobs In)\s+", "", s)
    for sfx in _SPAM_SUFFIXES:
        s = re.sub(rf"(?i){sfx}.*$", "", s)
    return s.strip()

def split_title_and_company(raw_title: str):
    cleaned   = _clean_raw_title(raw_title)
    job_title = cleaned
    company   = "Unknown"
    for sep in [" at ", " At ", " - ", " | ", " @ ", " – "]:
        if sep in cleaned:
            parts     = cleaned.split(sep, 1)
            job_title = parts[0].strip()
            company   = parts[1].strip()
            break
    job_title = re.sub(r"[^a-zA-Z0-9\s\+\#\./,]", "", job_title).strip().title()
    company   = re.sub(r"[^a-zA-Z0-9\s\.,&\-]",   "", company  ).strip().title()
    return (job_title or "Unknown"), (company or "Unknown")

# ============================================================
# SECTION 4: DATA EXTRACTORS
# ============================================================

def extract_emails(text: str) -> str:
    found = re.findall(
        r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,7}\b", text
    )
    _noise = {
        "noreply", "no-reply", "donotreply", "notifications", "mailer",
        "bounce", "alert", "privacy", "legal", "unsubscribe", "sentry",
        "example.com", "support", "help", "feedback", "abuse", "postmaster",
    }
    clean = [e for e in found if not any(n in e.lower() for n in _noise) and len(e) > 6]
    return ", ".join(sorted(set(clean))) if clean else "Not Found"

def detect_job_type(desc: str) -> str:
    d    = desc.lower()
    tags = []
    if re.search(r"\bc2c\b|corp[\s\-]?to[\s\-]?corp|corp2corp|c-2-c", d):
        tags.append("C2C")
    if re.search(r"\bw[\s\-]?2\b", d):
        tags.append("W2")
    if re.search(r"\b1099\b", d):
        tags.append("1099")
    if re.search(r"\bcontract[\s\-]to[\s\-]hire\b|\bc2h\b", d):
        tags.append("Contract-to-Hire")
    elif re.search(r"\bcontract\b|\bcontractor\b", d) and "Contract-to-Hire" not in tags:
        tags.append("Contract")
    if re.search(r"\bfull[\s\-]?time\b|\bpermanent\b|\bfte\b", d):
        tags.append("Full-Time")
    if re.search(r"\bpart[\s\-]?time\b", d):
        tags.append("Part-Time")
    if re.search(r"\bremote\b|work[\s\-]from[\s\-]home|\bwfh\b", d):
        tags.append("Remote")
    if re.search(r"\bhybrid\b", d):
        tags.append("Hybrid")
    if re.search(r"\bon[\s\-]?site\b|in[\s\-]?office\b|in[\s\-]?person\b", d):
        tags.append("Onsite")
    return ", ".join(tags) if tags else "Not Specified"

def extract_salary(desc: str) -> str:
    patterns = [
        r"\$[\d,]+(?:\.\d{2})?\s*[-–to]+\s*\$[\d,]+(?:\.\d{2})?(?:\s*(?:k|K|/hr|/hour|/year|/yr|annually|per\s+hour|per\s+annum))?",
        r"\$[\d,]+(?:\.\d{2})?(?:\s*(?:k|K|/hr|/hour|/year|/yr|annually|per\s+hour))",
        r"(?i)(?:salary|rate|compensation|pay)\s*[:\-]?\s*\$[\d,]+",
        r"[\d,]+\s*[-–]\s*[\d,]+\s*(?:USD|k|K)\s*(?:per\s+(?:year|hr|hour)|annually|/hr)",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return "Not Specified"

def extract_experience(desc: str) -> str:
    patterns = [
        r"\d+\+?\s*[-–to]+\s*\d+\s+years?\s+(?:of\s+)?(?:relevant\s+)?(?:professional\s+)?experience",
        r"\d+\+?\s+years?\s+(?:of\s+)?(?:relevant\s+)?(?:professional\s+)?experience",
        r"(?:minimum|min\.?|at\s+least)\s+\d+\+?\s+years?",
        r"\d+\+?\s+years?\s+(?:in|with|working\s+with)",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return "Not Specified"

def extract_location(desc: str) -> str:
    _states = (
        "AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|"
        "MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|"
        "TX|UT|VT|VA|WA|WV|WI|WY|DC"
    )
    patterns = [
        rf"(?:location|located\s+in|based\s+in|office\s+in)\s*[:\-]?\s*([A-Za-z\s]+,?\s*(?:{_states}|USA|United\s+States))",
        rf"([A-Za-z][A-Za-z\s]+,\s*(?:{_states}))\b",
        r"(?:Remote|Hybrid|Onsite)[,\s]+(?:in\s+)?([A-Za-z\s,]{4,40})",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m:
            loc = m.group(1).strip().rstrip(",").strip()
            if 2 < len(loc) < 60:
                return loc
    return "Not Specified"

_SKILLS = [
    (r"\bPython\b",                      "Python"),
    (r"\bScala\b",                       "Scala"),
    (r"\bJava\b",                        "Java"),
    (r"\b(?:SQL|T-SQL|PL/SQL)\b",        "SQL"),
    (r"\bGo\b|\bGolang\b",               "Go"),
    (r"\bShell\b|\bBash\b",              "Shell/Bash"),
    (r"\bPySpark\b",                     "PySpark"),
    (r"\b(?:Apache\s+)?Spark\b",         "Spark"),
    (r"\b(?:Apache\s+)?Kafka\b",         "Kafka"),
    (r"\b(?:Apache\s+)?Flink\b",         "Flink"),
    (r"\b(?:Apache\s+)?Hadoop\b",        "Hadoop"),
    (r"\bHive\b",                        "Hive"),
    (r"\bKinesis\b",                     "Kinesis"),
    (r"\bAWS\b|Amazon\s+Web\s+Services", "AWS"),
    (r"\bAzure\b|Microsoft\s+Azure",     "Azure"),
    (r"\bGCP\b|Google\s+Cloud",          "GCP"),
    (r"\bDatabricks\b",                  "Databricks"),
    (r"\bSnowflake\b",                   "Snowflake"),
    (r"\bRedshift\b",                    "Redshift"),
    (r"\bBigQuery\b",                    "BigQuery"),
    (r"\bSynapse\b",                     "Synapse"),
    (r"\bDelta\s+Lake\b",                "Delta Lake"),
    (r"\bIceberg\b",                     "Iceberg"),
    (r"\bParquet\b",                     "Parquet"),
    (r"\bAirflow\b",                     "Airflow"),
    (r"\bdbt\b|data\s+build\s+tool",     "dbt"),
    (r"\bPrefect\b",                     "Prefect"),
    (r"\bDagster\b",                     "Dagster"),
    (r"\bGlue\b",                        "AWS Glue"),
    (r"\bTerraform\b",                   "Terraform"),
    (r"\bDocker\b",                      "Docker"),
    (r"\bKubernetes\b|\bK8s\b",          "Kubernetes"),
    (r"\bGit\b",                         "Git"),
    (r"\bPostgreSQL\b|\bpostgres\b",     "PostgreSQL"),
    (r"\bMySQL\b",                       "MySQL"),
    (r"\bMongoDB\b",                     "MongoDB"),
    (r"\bDynamoDB\b",                    "DynamoDB"),
    (r"\bRedis\b",                       "Redis"),
    (r"\bMLflow\b",                      "MLflow"),
    (r"\bSageMaker\b",                   "SageMaker"),
    (r"\bPower\s+BI\b",                  "Power BI"),
    (r"\bTableau\b",                     "Tableau"),
    (r"\bLooker\b",                      "Looker"),
]

def extract_skills(desc: str) -> str:
    found = []
    for pattern, label in _SKILLS:
        if re.search(pattern, desc, re.IGNORECASE) and label not in found:
            found.append(label)
    return ", ".join(found[:30]) if found else "Not Specified"

# ============================================================
# SECTION 5: curl_cffi SCRAPER (Chrome impersonation)
# ============================================================

def _make_session():
    """Create a curl_cffi session that impersonates Chrome 131."""
    if _HAS_CURL_CFFI:
        return cf_requests.Session(impersonate="chrome131")
    # Fallback: plain requests session with browser headers
    import requests
    s = requests.Session()
    s.headers.update({
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
    })
    return s

_SCRAPE_SESSION = _make_session()   # shared session for connection pooling

_CONTENT_IDS = [
    "content", "job-description", "jobDescription",
    "job_description", "job-content", "main-content",
]
_CONTENT_CLASSES = [
    re.compile(p, re.IGNORECASE) for p in [
        r"job[\-_]?description", r"posting[\-_]?description",
        r"job[\-_]?details", r"position[\-_]?description",
        r"\bjob[\-_]?content\b", r"\bdescription\b", r"\bcontent\b",
    ]
]
_BOT_SIGNALS = [
    "checking your browser", "access denied", "cloudflare",
    "captcha", "enable javascript", "robot check", "just a moment",
]

def _extract_main_content(soup: BeautifulSoup) -> str:
    for id_val in _CONTENT_IDS:
        el = soup.find(id=id_val)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    for pat in _CONTENT_CLASSES:
        el = soup.find(class_=pat)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    for tag in ["main", "article"]:
        el = soup.find(tag)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)
    return soup.get_text(separator=" ", strip=True)

def scrape_url(url: str, fallback_snippet: str = "") -> tuple:
    """
    Returns: (page_title, description, was_scraped)
    Uses curl_cffi Chrome impersonation to bypass 403 on job portals.
    """
    if is_snippet_only_url(url):
        return "", fallback_snippet, False

    for attempt in range(2):
        try:
            resp = _SCRAPE_SESSION.get(
                url,
                timeout=CONFIG["http_timeout"],
                allow_redirects=True,
            )
            if resp.status_code in (403, 429, 503):
                time.sleep(random.uniform(4, 8))
                continue
            if resp.status_code != 200:
                return "", fallback_snippet, False

            soup = BeautifulSoup(resp.text, "lxml")
            for tag in soup(["script","style","footer","nav","header","aside","noscript","iframe"]):
                tag.decompose()

            page_title = ""
            if soup.title and soup.title.string:
                page_title = soup.title.string.strip()

            raw_text    = _extract_main_content(soup)
            description = re.sub(r"\s+", " ", raw_text).strip()

            if any(sig in description.lower() for sig in _BOT_SIGNALS):
                return "", fallback_snippet, False
            if len(description) < 150:
                return page_title, fallback_snippet, False

            return page_title, description[:8000], True

        except Exception as exc:
            log.debug("Scrape error [%s]: %s", url[:60], str(exc)[:60])
            break

    return "", fallback_snippet, False

# ============================================================
# SECTION 6: LINK VALIDATOR
# ============================================================

_EXPIRED_SIGNALS = [
    "job no longer available", "position has been filled",
    "this job has expired", "no longer accepting applications",
    "job has been closed", "listing has expired", "role has been filled",
    "position is filled", "job has been removed",
    "opportunity has been filled", "job is closed", "posting is closed",
]

def validate_link_status(description: str, was_scraped: bool) -> str:
    if not was_scraped:
        return "Unverified"
    if not description or len(description) < 100:
        return "Unknown"
    low = description.lower()
    if any(s in low for s in _EXPIRED_SIGNALS):
        return "Expired"
    return "Active"

# ============================================================
# SECTION 7: RESULT PROCESSOR
# ============================================================

def process_result(res: dict, canonical_role: str) -> dict | None:
    """Convert one DDGS result dict into a validated job record, or None."""
    url     = (res.get("href") or "").strip()
    snippet = (res.get("body") or "")
    s_title = (res.get("title") or "")

    if not url or is_garbage_url(url):
        return None
    if not url.startswith("http"):
        url = "https://" + url

    page_title, description, was_scraped = scrape_url(url, snippet)

    # Use snippet if page description is too thin
    if len(description.strip()) < 200:
        description = snippet

    # Hard reject: not English / not a job page
    if not is_real_job_page(description):
        log.debug("⛔ Not a job page, skipping: %s", url[:70])
        return None

    raw_title          = page_title if page_title else s_title
    job_title, company = split_title_and_company(raw_title)
    if not job_title or len(job_title.strip()) < 2:
        job_title = canonical_role.title()

    link_status = validate_link_status(description, was_scraped)
    if link_status == "Expired":
        log.info("⏩ Expired — skipping: %s", url[:70])
        return None

    now = datetime.now()
    rec = {
        "id":                  str(uuid.uuid4()),
        "job_hash":            hashlib.md5(url.encode()).hexdigest(),
        "created_at":          now,
        "updated_at":          now,
        "fetch_date":          now.date(),
        "search_keyword":      canonical_role,
        "company_name":        company,
        "job_title":           job_title,
        "job_description":     description,
        "apply_link":          url,
        "hr_email":            extract_emails(description),
        "job_type":            detect_job_type(description),
        "salary_range":        extract_salary(description),
        "experience_required": extract_experience(description),
        "location":            extract_location(description),
        "skills_required":     extract_skills(description),
        "link_status":         link_status,
        "validation_score":    0,          # computed below
        "validation_status":   "Pending",
    }
    score                    = compute_validation_score(rec)
    rec["validation_score"]  = score
    rec["validation_status"] = get_validation_status(score)

    # Drop records below minimum quality threshold
    if score < CONFIG["min_score"]:
        log.debug("⛔ Score %d < %d — dropping: %s", score, CONFIG["min_score"], url[:70])
        return None

    return rec

# ============================================================
# SECTION 8: SEARCH ENGINE (DDGS with backend rotation)
# ============================================================

def _ddgs_search(query: str) -> list:
    """
    Try GOOD_BACKENDS in order.
    — Never falls back to mojeek/grokipedia/yahoo for site: queries.
    — On 403 / rate-limit: sleep longer, try next backend.
    """
    for backend in GOOD_BACKENDS:
        try:
            with DDGS() as ddgs:
                results = list(ddgs.text(
                    query,
                    backend=backend,
                    region="us-en",
                    max_results=CONFIG["max_results"],
                    timelimit=CONFIG["time_filter"],
                ))
            if results:
                log.debug("  ✔ backend=%s → %d hits", backend, len(results))
                return results
        except Exception as exc:
            err = str(exc)
            if "403" in err or "Ratelimit" in err or "ratelimit" in err.lower():
                sleep_t = random.uniform(*CONFIG["ratelimit_sleep"])
                log.warning("  ⚠️  Rate-limited on backend=%s — sleeping %.1fs", backend, sleep_t)
                time.sleep(sleep_t)
            elif "No results" not in err:
                log.debug("  backend=%s error: %s", backend, err[:80])
    return []

def search_for_role(canonical_role: str, phrases: list) -> list:
    """
    Multi-portal search for one role.
    For each portal we try the phrase list and take the first that returns results.
    """
    records:   list = []
    seen_urls: set  = set()

    for portal in PORTAL_SITES:
        # Try each phrase alias until one works
        raw_results = []
        for phrase in phrases:
            # Short focused query — avoids 237-char OR blobs that confuse engines
            query = f'{portal} {phrase} job USA'
            log.info("  🔍 %s", query)

            raw_results = _ddgs_search(query)
            if raw_results:
                break
            time.sleep(random.uniform(2, 3))   # brief gap between phrase retries

        if not raw_results:
            time.sleep(random.uniform(*CONFIG["search_delay"]))
            continue

        log.info("    📥 %d raw results — scraping …", len(raw_results))

        new_results = [r for r in raw_results if r.get("href","") not in seen_urls]
        with ThreadPoolExecutor(max_workers=CONFIG["scrape_workers"]) as pool:
            futures = {pool.submit(process_result, res, canonical_role): res
                       for res in new_results}
            try:
                for future in as_completed(futures, timeout=90):
                    try:
                        job = future.result()
                        if job and job["apply_link"] not in seen_urls:
                            seen_urls.add(job["apply_link"])
                            records.append(job)
                    except Exception as exc:
                        log.debug("Future error: %s", str(exc)[:60])
            except FutureTimeout:
                log.warning("  ⚠️  Some futures timed out — continuing")

        for r in raw_results:
            seen_urls.add(r.get("href", ""))

        time.sleep(random.uniform(*CONFIG["search_delay"]))   # polite gap between portals

    log.info("  ✅ %s → %d valid jobs", canonical_role, len(records))
    return records

# ============================================================
# SECTION 9: DELTA LAKE WRITER
# ============================================================

JOB_SCHEMA = StructType([
    StructField("id",                  StringType(),    True),
    StructField("job_hash",            StringType(),    True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("fetch_date",          DateType(),      True),
    StructField("search_keyword",      StringType(),    True),
    StructField("company_name",        StringType(),    True),
    StructField("job_title",           StringType(),    True),
    StructField("job_description",     StringType(),    True),
    StructField("apply_link",          StringType(),    True),
    StructField("hr_email",            StringType(),    True),
    StructField("job_type",            StringType(),    True),
    StructField("salary_range",        StringType(),    True),
    StructField("experience_required", StringType(),    True),
    StructField("location",            StringType(),    True),
    StructField("skills_required",     StringType(),    True),
    StructField("link_status",         StringType(),    True),
    StructField("validation_score",    IntegerType(),   True),
    StructField("validation_status",   StringType(),    True),
])

_NEW_COLUMNS_V7 = [
    ("hr_email",            "STRING"),
    ("job_type",            "STRING"),
    ("salary_range",        "STRING"),
    ("experience_required", "STRING"),
    ("location",            "STRING"),
    ("skills_required",     "STRING"),
    ("link_status",         "STRING"),
    ("validation_score",    "INT"),
]

def ensure_table(spark: SparkSession, table: str):
    """Create catalog/schema/table if needed, add missing V7 columns."""
    parts = table.split(".")
    if len(parts) == 3:
        catalog, schema, _ = parts
        for ddl in [
            f"CREATE CATALOG IF NOT EXISTS `{catalog}`",
            f"CREATE SCHEMA  IF NOT EXISTS `{catalog}`.`{schema}`",
        ]:
            try:
                spark.sql(ddl)
            except Exception:
                pass

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id                  STRING,
            job_hash            STRING,
            created_at          TIMESTAMP,
            updated_at          TIMESTAMP,
            fetch_date          DATE,
            search_keyword      STRING,
            company_name        STRING,
            job_title           STRING,
            job_description     STRING,
            apply_link          STRING,
            hr_email            STRING,
            job_type            STRING,
            salary_range        STRING,
            experience_required STRING,
            location            STRING,
            skills_required     STRING,
            link_status         STRING,
            validation_score    INT,
            validation_status   STRING
        )
        USING DELTA
        PARTITIONED BY (fetch_date)
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    # Schema migration: add V7 columns to older tables
    try:
        existing = {c.lower() for c in spark.table(table).columns}
        for col_name, col_type in _NEW_COLUMNS_V7:
            if col_name.lower() not in existing:
                spark.sql(f"ALTER TABLE {table} ADD COLUMN {col_name} {col_type}")
                log.info("🔧 Added column: %s %s", col_name, col_type)
    except Exception as exc:
        log.warning("Schema migration (non-fatal): %s", exc)

def write_to_delta(records: list, spark: SparkSession) -> int:
    if not records:
        log.warning("No records to write.")
        return 0

    table = CONFIG["delta_table"]
    ensure_table(spark, table)

    df    = spark.createDataFrame(records, schema=JOB_SCHEMA)
    df    = df.dropDuplicates(["job_hash"])
    count = df.count()

    (
        DeltaTable.forName(spark, table).alias("tgt")
        .merge(df.alias("src"), "tgt.job_hash = src.job_hash")
        .whenMatchedUpdate(set={
            "updated_at":          "src.updated_at",
            "link_status":         "src.link_status",
            "hr_email":            "src.hr_email",
            "job_description":     "src.job_description",
            "job_type":            "src.job_type",
            "salary_range":        "src.salary_range",
            "skills_required":     "src.skills_required",
            "experience_required": "src.experience_required",
            "location":            "src.location",
            "validation_score":    "src.validation_score",
            "validation_status":   "src.validation_status",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )
    return count

# ============================================================
# SECTION 10: MAIN ORCHESTRATOR
# ============================================================

def run_harvester():
    log.info("=" * 65)
    log.info("🚀 US IT JOB HARVESTER V7 — BROWSER-GRADE")
    log.info("📅 Run date      : %s", date.today())
    log.info("🎯 Role groups   : %d  (with aliases)", len(ROLES))
    log.info("🔗 Portals       : %d", len(PORTAL_SITES))
    log.info("⏱️  Time filter   : last-%s",
             {"d":"day","w":"week","m":"month"}.get(CONFIG["time_filter"], CONFIG["time_filter"]))
    log.info("🌐 HTTP backend  : %s",
             "curl_cffi Chrome131" if _HAS_CURL_CFFI else "requests (install curl_cffi for better results)")
    log.info("🔍 Search engine : duckduckgo → startpage → brave (mojeek NEVER used)")
    log.info("=" * 65)

    all_records: list = []

    for canonical_role, phrases in ROLES.items():
        log.info("\n📌 Role: %s  (aliases: %s)", canonical_role, " | ".join(phrases))
        try:
            role_jobs = search_for_role(canonical_role, phrases)
            all_records.extend(role_jobs)
        except Exception as exc:
            log.error("❌ Error for '%s': %s", canonical_role, exc)
        time.sleep(random.uniform(5.0, 9.0))   # gap between role groups

    # Global dedup by job_hash
    seen:   set  = set()
    unique: list = []
    for rec in all_records:
        h = rec["job_hash"]
        if h not in seen:
            seen.add(h)
            unique.append(rec)

    log.info("\n📊 Collected (all roles)    : %d", len(all_records))
    log.info("📊 After global dedup       : %d", len(unique))
    log.info("📊 Valid   (score ≥ 70)     : %d", sum(1 for r in unique if r["validation_score"] >= 70))
    log.info("📊 Partial (score 45-69)    : %d", sum(1 for r in unique if 45 <= r["validation_score"] < 70))

    written = write_to_delta(unique, spark)

    log.info("\n" + "=" * 65)
    log.info("🎉 HARVEST COMPLETE — %d records written", written)
    log.info("=" * 65)

    # Summary table in notebook output
    spark.sql(f"""
        SELECT
            search_keyword,
            validation_status,
            COUNT(*)                                              AS jobs,
            ROUND(AVG(validation_score))                         AS avg_score,
            COUNT(CASE WHEN hr_email    != 'Not Found'     THEN 1 END) AS with_email,
            COUNT(CASE WHEN salary_range!= 'Not Specified' THEN 1 END) AS with_salary,
            COUNT(CASE WHEN job_type   != 'Not Specified'  THEN 1 END) AS with_type
        FROM {CONFIG['delta_table']}
        WHERE fetch_date = current_date()
        GROUP BY search_keyword, validation_status
        ORDER BY search_keyword, avg_score DESC
    """).show(50, truncate=False)

# ============================================================
# ▶️  ENTRY POINT
# ============================================================
run_harvester()


In [0]:
%sql
select * from main.jobs_automation.raw_jobs_staging

In [0]:
%sql
select * from jobs_automation_db.default.validated_jobs_master

In [0]:
%sql
delete from main.jobs_automation.raw_jobs_staging

In [0]:
%skip
%sql
select * from main.jobs_automation.raw_jobs_staging

In [0]:
# ============================================================
# US IT JOB HARVESTER — V6  (Production Grade)
# Databricks | Delta Lake | Daily Schedule Ready
# ============================================================
# What's new vs V5:
#   ✅ HR email extraction (filters noise/noreply addresses)
#   ✅ Job type detection  (C2C / W2 / 1099 / Contract / FT / Remote)
#   ✅ Salary range extraction
#   ✅ Skills extraction   (PySpark, Spark, Kafka, AWS, dbt …)
#   ✅ Experience required extraction
#   ✅ Location extraction from page content
#   ✅ Link validation     (Active / Expired / Unverified / Unknown)
#   ✅ Smart content selector (greenhouse, lever, workday …)
#   ✅ Rotating user-agents for anti-bot
#   ✅ Graceful retry + timeout on every scrape
#   ✅ Full dedup by job_hash before Delta merge
#   ✅ Upsert — refreshes stale records on re-run
#   ✅ Schema migration helper (adds new columns to old table)
#   ✅ Unity Catalog compatible (catalog.schema.table)
# ============================================================

import hashlib
import uuid
import re
import time
import random
import requests
from bs4 import BeautifulSoup
from datetime import date, datetime
from ddgs import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, TimestampType,
)
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError as FutureTimeout
import warnings
import logging

warnings.filterwarnings("ignore", category=ResourceWarning)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ============================================================
# ⚙️  CONFIG  — tune this section before scheduling
# ============================================================
CONFIG = {
    # Roles to harvest
    "roles": [
        "Data Engineer",
        "PySpark Developer",
        "Spark Developer",
        "ETL Developer",
        "Data Platform Engineer",
        "Analytics Engineer",
        "Big Data Engineer",
        "Cloud Data Engineer",
        "Databricks Engineer",
        "AWS Data Engineer",
        "Python Developer",
    ],
    # US locations
    "locations": [
        "USA", "Remote",
        "Texas", "New York", "California", "Illinois",
        "Georgia", "New Jersey", "Florida", "Washington",
        "Chicago", "Dallas", "Austin", "Atlanta", "Seattle",
    ],
    # Unity-catalog qualified table  (catalog.schema.table)
    "delta_table": "main.jobs_automation.raw_jobs_staging",
    # Max results per DDGS query per site
    "max_results": 15,
    # Concurrent scrape workers per site batch
    "scrape_workers": 5,
    # DDGS timelimit: 'd' = last day  |  'w' = last week
    "time_filter": "d",
    # HTTP request timeout (seconds)
    "http_timeout": 10,
}

# ---------------------------------------------------------------------------
# Job portals — site: operator targets only job-posting pages
# ---------------------------------------------------------------------------
PORTAL_SITES = [
    "site:dice.com/job-detail",
    "site:wellfound.com/jobs",
    "site:weworkremotely.com/remote-jobs",
    "site:remoteok.com",
    "site:greenhouse.io",
    "site:boards.greenhouse.io",
    "site:jobs.lever.co",
    "site:myworkdayjobs.com",
    "site:jobs.ashbyhq.com",
    "site:apply.workable.com",
    "site:smartrecruiters.com/jobs",
    "site:linkedin.com/jobs/view",
    "site:simplyhired.com/job",
    "site:indeed.com/viewjob",
    "site:ziprecruiter.com/jobs",
    "site:jobvite.com/job",
]

# Sites that block scraping entirely → we use the DDGS snippet only
SNIPPET_ONLY: set = {
    "linkedin.com", "dice.com", "wellfound.com",
    "ziprecruiter.com", "monster.com", "indeed.com",
    "glassdoor.com", "careerbuilder.com", "hiringcafe.com",
    "simplyhired.com",
}

# Non-job domains to ignore completely
GARBAGE_DOMAINS: set = {
    "wikipedia.org", "w3schools.com", "python.org", "geeksforgeeks.org",
    "youtube.com", "github.com/topics", "stackoverflow.com", "medium.com",
    "dev.to", "reddit.com", "quora.com", "coursera.org", "udemy.com",
    "zhihu.com", "taptap.io", "codecademy.com", "pluralsight.com",
    "tutorialspoint.com", "hackerearth.com", "ibm.com/docs",
    "glassdoor.com/interview", "glassdoor.com/learn",
}

# Rotating user-agents to avoid bot detection
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "Chrome/122.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "Chrome/121.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:124.0) Gecko/20100101 "
    "Firefox/124.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "Chrome/122.0.0.0 Safari/537.36",
]

# ============================================================
# SECTION 1 — URL FILTERS
# ============================================================

def is_garbage_url(url: str) -> bool:
    low = url.lower()
    return any(d in low for d in GARBAGE_DOMAINS)

def is_snippet_only_url(url: str) -> bool:
    low = url.lower()
    return any(d in low for d in SNIPPET_ONLY)


# ============================================================
# SECTION 2 — TITLE / COMPANY CLEANER
# ============================================================

_SPAM_SUFFIXES = [
    r" - LinkedIn", r" \| LinkedIn", r" - Greenhouse", r" - Lever",
    r" \| Wellfound", r" - Dice\.com", r" \| Built In", r" - Indeed",
    r" - ZipRecruiter", r" \| Glassdoor", r" - Monster", r" - Remote OK",
    r" - Jobvite", r" - SmartRecruiters", r" \| Workable",
]

def _clean_raw_title(raw: str) -> str:
    s = re.sub(r"\[.*?\]|\(.*?\)", "", raw)
    s = re.sub(r"(?i)(Job Application for|Apply for|Jobs In)\s+", "", s)
    for suffix in _SPAM_SUFFIXES:
        s = re.sub(rf"(?i){suffix}.*$", "", s)
    return s.strip()

def split_title_and_company(raw_title: str):
    """Returns (job_title, company_name) from a raw page/search title."""
    cleaned = _clean_raw_title(raw_title)
    job_title, company = cleaned, "Unknown"
    for sep in [" at ", " At ", " - ", " | ", " @ ", " – "]:
        if sep in cleaned:
            parts = cleaned.split(sep, 1)
            job_title = parts[0].strip()
            company   = parts[1].strip()
            break
    job_title = re.sub(r"[^a-zA-Z0-9\s\+\#\./,]", "", job_title).strip().title()
    company   = re.sub(r"[^a-zA-Z0-9\s\.,&\-]",   "", company  ).strip().title()
    return (job_title or "Unknown"), (company or "Unknown")


# ============================================================
# SECTION 3 — DATA EXTRACTORS
# ============================================================

def extract_emails(text: str) -> str:
    """Pull valid recruiter/HR emails, strip noise addresses."""
    found = re.findall(
        r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,7}\b", text
    )
    _noise = {
        "noreply", "no-reply", "donotreply", "notifications", "mailer",
        "bounce", "alert", "privacy", "legal", "unsubscribe", "sentry",
        "example.com", "support", "help", "feedback", "abuse", "postmaster",
    }
    clean = [
        e for e in found
        if not any(n in e.lower() for n in _noise) and len(e) > 6
    ]
    return ", ".join(sorted(set(clean))) if clean else "Not Found"


def detect_job_type(desc: str) -> str:
    """Detect C2C / W2 / 1099 / Contract / Full-Time / Remote / Hybrid / Onsite."""
    d = desc.lower()
    tags = []

    # --- Tax / engagement type ---
    if re.search(r"\bc2c\b|corp[\s\-]?to[\s\-]?corp|corp2corp|c-2-c", d):
        tags.append("C2C")
    if re.search(r"\bw[\s\-]?2\b", d):
        tags.append("W2")
    if re.search(r"\b1099\b", d):
        tags.append("1099")
    if re.search(r"\bcontract[\s\-]to[\s\-]hire\b|\bc2h\b", d):
        tags.append("Contract-to-Hire")
    elif re.search(r"\bcontract\b|\bcontractor\b", d) and "Contract-to-Hire" not in tags:
        tags.append("Contract")
    if re.search(r"\bfull[\s\-]?time\b|\bpermanent\b|\bfte\b", d):
        tags.append("Full-Time")
    if re.search(r"\bpart[\s\-]?time\b", d):
        tags.append("Part-Time")

    # --- Work model ---
    if re.search(r"\bremote\b|work[\s\-]from[\s\-]home|\bwfh\b", d):
        tags.append("Remote")
    if re.search(r"\bhybrid\b", d):
        tags.append("Hybrid")
    if re.search(r"\bon[\s\-]?site\b|in[\s\-]?office\b|in[\s\-]?person\b", d):
        tags.append("Onsite")

    return ", ".join(tags) if tags else "Not Specified"


def extract_salary(desc: str) -> str:
    """Extract salary / hourly rate range."""
    patterns = [
        # $120,000 – $160,000/year  or  $60/hr
        r"\$[\d,]+(?:\.\d{2})?\s*[-–to]+\s*\$[\d,]+(?:\.\d{2})?(?:\s*(?:k|K|/hr|/hour|/year|/yr|annually|per\s+hour|per\s+annum))?",
        r"\$[\d,]+(?:\.\d{2})?(?:\s*(?:k|K|/hr|/hour|/year|/yr|annually|per\s+hour))",
        r"(?i)(?:salary|rate|compensation|pay)\s*[:\-]?\s*\$[\d,]+",
        r"[\d,]+\s*[-–]\s*[\d,]+\s*(?:USD|dollars|k|K)?\s*(?:per\s+(?:year|hr|hour)|annually|/hr)",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return "Not Specified"


def extract_experience(desc: str) -> str:
    """Extract required years of experience."""
    patterns = [
        r"\d+\+?\s*[-–to]+\s*\d+\s+years?\s+(?:of\s+)?(?:relevant\s+)?(?:professional\s+)?experience",
        r"\d+\+?\s+years?\s+(?:of\s+)?(?:relevant\s+)?(?:professional\s+)?experience",
        r"(?:minimum|min\.?|at\s+least)\s+\d+\+?\s+years?",
        r"\d+\+?\s+years?\s+(?:in|with|working\s+with)",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m:
            return m.group(0).strip()
    return "Not Specified"


def extract_location(desc: str) -> str:
    """Extract job location from description text."""
    _states = (
        "AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|ID|IL|IN|IA|KS|KY|LA|ME|MD|MA|"
        "MI|MN|MS|MO|MT|NE|NV|NH|NJ|NM|NY|NC|ND|OH|OK|OR|PA|RI|SC|SD|TN|"
        "TX|UT|VT|VA|WA|WV|WI|WY|DC"
    )
    patterns = [
        rf"(?:location|located\s+in|based\s+in|office\s+in)\s*[:\-]?\s*([A-Za-z\s]+,?\s*(?:{_states}|USA|United\s+States))",
        rf"([A-Za-z][A-Za-z\s]+,\s*(?:{_states}))\b",
        r"(?:Remote|Hybrid|Onsite)[,\s]+(?:in\s+)?([A-Za-z\s,]{4,40})",
    ]
    for pat in patterns:
        m = re.search(pat, desc, re.IGNORECASE)
        if m:
            loc = m.group(1).strip().rstrip(",").strip()
            if 2 < len(loc) < 60:
                return loc
    return "Not Specified"


# Skill patterns: (regex, canonical label)
_SKILLS = [
    # Languages
    (r"\bPython\b",                     "Python"),
    (r"\bScala\b",                      "Scala"),
    (r"\bJava\b",                       "Java"),
    (r"\b(?:SQL|T-SQL|PL/SQL)\b",       "SQL"),
    (r"\bGo\b|\bGolang\b",              "Go"),
    (r"\bRust\b",                       "Rust"),
    (r"\bKotlin\b",                     "Kotlin"),
    (r"\bShell\b|\bBash\b",             "Shell/Bash"),
    # Big Data
    (r"\bPySpark\b",                    "PySpark"),
    (r"\b(?:Apache\s+)?Spark\b",        "Spark"),
    (r"\b(?:Apache\s+)?Kafka\b",        "Kafka"),
    (r"\b(?:Apache\s+)?Flink\b",        "Flink"),
    (r"\b(?:Apache\s+)?Hadoop\b",       "Hadoop"),
    (r"\bHive\b",                       "Hive"),
    (r"\bKinesis\b",                    "Kinesis"),
    (r"\bNiFi\b",                       "NiFi"),
    # Cloud
    (r"\bAWS\b|Amazon\s+Web\s+Services","AWS"),
    (r"\bAzure\b|Microsoft\s+Azure",    "Azure"),
    (r"\bGCP\b|Google\s+Cloud",         "GCP"),
    # Lakehouse / Warehouse
    (r"\bDatabricks\b",                 "Databricks"),
    (r"\bSnowflake\b",                  "Snowflake"),
    (r"\bRedshift\b",                   "Redshift"),
    (r"\bBigQuery\b",                   "BigQuery"),
    (r"\bSynapse\b",                    "Synapse"),
    (r"\bDelta\s+Lake\b",               "Delta Lake"),
    (r"\bIceberg\b",                    "Iceberg"),
    (r"\bParquet\b",                    "Parquet"),
    # Orchestration
    (r"\bAirflow\b",                    "Airflow"),
    (r"\bdbt\b|data\s+build\s+tool",    "dbt"),
    (r"\bPrefect\b",                    "Prefect"),
    (r"\bDagster\b",                    "Dagster"),
    (r"\bGlue\b",                       "AWS Glue"),
    # DevOps / Infra
    (r"\bTerraform\b",                  "Terraform"),
    (r"\bDocker\b",                     "Docker"),
    (r"\bKubernetes\b|\bK8s\b",         "Kubernetes"),
    (r"\bGit\b",                        "Git"),
    (r"\bJenkins\b",                    "Jenkins"),
    (r"\bCI/CD\b",                      "CI/CD"),
    # Databases
    (r"\bPostgreSQL\b|\bpostgres\b",    "PostgreSQL"),
    (r"\bMySQL\b",                      "MySQL"),
    (r"\bMongoDB\b",                    "MongoDB"),
    (r"\bCassandra\b",                  "Cassandra"),
    (r"\bDynamoDB\b",                   "DynamoDB"),
    (r"\bRedis\b",                      "Redis"),
    # ML / Misc
    (r"\bMLflow\b",                     "MLflow"),
    (r"\bSageMaker\b",                  "SageMaker"),
    (r"\bPower\s+BI\b",                 "Power BI"),
    (r"\bTableau\b",                    "Tableau"),
    (r"\bLooker\b",                     "Looker"),
]

def extract_skills(desc: str) -> str:
    found = []
    for pattern, label in _SKILLS:
        if re.search(pattern, desc, re.IGNORECASE) and label not in found:
            found.append(label)
    return ", ".join(found[:30]) if found else "Not Specified"


# ============================================================
# SECTION 4 — PAGE SCRAPER
# ============================================================

# Content selectors ordered by specificity (ATS-specific first)
_CONTENT_IDS = [
    "content", "job-description", "jobDescription",
    "job_description", "job-content", "main-content",
]
_CONTENT_CLASSES = [
    re.compile(p, re.IGNORECASE)
    for p in [
        r"job[\-_]?description", r"jobDescription", r"posting[\-_]?description",
        r"job[\-_]?details", r"job[\-_]?content", r"position[\-_]?description",
        r"\bdescription\b", r"\bcontent\b",
    ]
]

def _extract_main_content(soup: BeautifulSoup) -> str:
    # 1. Try ID selectors
    for id_val in _CONTENT_IDS:
        el = soup.find(id=id_val)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)

    # 2. Try class selectors (compiled regex)
    for pat in _CONTENT_CLASSES:
        el = soup.find(class_=pat)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)

    # 3. Semantic tags
    for tag in ["main", "article"]:
        el = soup.find(tag)
        if el and len(el.get_text(strip=True)) > 200:
            return el.get_text(separator=" ", strip=True)

    # 4. Full-body fallback
    return soup.get_text(separator=" ", strip=True)


_BOT_SIGNALS = [
    "checking your browser", "access denied", "cloudflare",
    "captcha", "enable javascript", "robot check", "just a moment",
]

def scrape_url(url: str, fallback_snippet: str = "") -> tuple:
    """
    Scrape a job URL.
    Returns: (page_title: str, description: str, was_scraped: bool)
    """
    if is_snippet_only_url(url):
        return "", fallback_snippet, False

    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Cache-Control": "no-cache",
    }

    for attempt in range(2):
        try:
            resp = requests.get(
                url, headers=headers,
                timeout=CONFIG["http_timeout"],
                allow_redirects=True,
            )
            if resp.status_code in (403, 429, 503):
                time.sleep(random.uniform(4, 7))
                continue
            if resp.status_code != 200:
                return "", fallback_snippet, False

            soup = BeautifulSoup(resp.text, "lxml")

            # Strip noise nodes
            for tag in soup(["script", "style", "footer", "nav",
                              "header", "aside", "noscript", "iframe"]):
                tag.decompose()

            page_title = ""
            if soup.title and soup.title.string:
                page_title = soup.title.string.strip()

            raw_text = _extract_main_content(soup)
            description = re.sub(r"\s+", " ", raw_text).strip()

            # Bot-page check
            desc_low = description.lower()
            if any(sig in desc_low for sig in _BOT_SIGNALS):
                return "", fallback_snippet, False

            if len(description) < 150:
                return page_title, fallback_snippet, False

            return page_title, description[:8000], True

        except requests.exceptions.Timeout:
            log.debug("Timeout: %s", url)
            break
        except Exception as exc:
            log.debug("Scrape error [%s]: %s", url[:60], str(exc)[:60])
            break

    return "", fallback_snippet, False


# ============================================================
# SECTION 5 — LINK VALIDATOR
# ============================================================

_EXPIRED_SIGNALS = [
    "job no longer available", "position has been filled",
    "this job has expired", "no longer accepting applications",
    "job has been closed", "listing has expired", "role has been filled",
    "position is filled", "job has been removed", "this position has been",
    "opportunity has been filled", "job is closed", "vacancy is closed",
    "application period has ended", "posting is closed",
]

def validate_link_status(description: str, was_scraped: bool) -> str:
    """
    Returns:
        'Active'      — page loaded, no expiry signals
        'Expired'     — page explicitly says position is closed
        'Unverified'  — blocked site, only have snippet
        'Unknown'     — page loaded but content too thin
    """
    if not was_scraped:
        return "Unverified"
    if not description or len(description) < 100:
        return "Unknown"
    low = description.lower()
    if any(sig in low for sig in _EXPIRED_SIGNALS):
        return "Expired"
    return "Active"


# ============================================================
# SECTION 6 — SINGLE RESULT PROCESSOR
# ============================================================

def process_result(res: dict, role: str):
    """Convert one DDGS result dict into a structured job record dict, or None."""
    url     = (res.get("href") or "").strip()
    snippet = (res.get("body") or "")
    s_title = (res.get("title") or "")

    if not url or is_garbage_url(url):
        return None

    if not url.startswith("http"):
        url = "https://" + url

    page_title, description, was_scraped = scrape_url(url, snippet)

    # Use snippet if scrape description is too thin
    if len(description.strip()) < 200:
        description = snippet

    raw_title = page_title if page_title else s_title
    job_title, company = split_title_and_company(raw_title)
    if not job_title or len(job_title.strip()) < 2:
        job_title = role.title()

    link_status = validate_link_status(description, was_scraped)
    if link_status == "Expired":
        log.info("⏩ Expired — skipping: %s", url[:80])
        return None

    now = datetime.now()
    return {
        "id":                  str(uuid.uuid4()),
        "job_hash":            hashlib.md5(url.encode()).hexdigest(),
        "created_at":          now,
        "updated_at":          now,
        "fetch_date":          now.date(),
        "search_keyword":      role,
        "company_name":        company,
        "job_title":           job_title,
        "job_description":     description,
        "apply_link":          url,
        "hr_email":            extract_emails(description),
        "job_type":            detect_job_type(description),
        "salary_range":        extract_salary(description),
        "experience_required": extract_experience(description),
        "location":            extract_location(description),
        "skills_required":     extract_skills(description),
        "link_status":         link_status,
        "validation_status":   "Pending",
    }


# ============================================================
# SECTION 7 — SEARCH ENGINE
# ============================================================

def search_for_role(role: str) -> list:
    """
    Run multi-portal DDGS search for one role, scrape each result
    concurrently, return list of clean job dicts.
    """
    loc_query = " OR ".join(f'"{loc}"' for loc in CONFIG["locations"])
    records:   list = []
    seen_urls: set  = set()

    with DDGS() as ddgs:
        for site in PORTAL_SITES:
            query = f'{site} "{role}" ({loc_query})'
            log.info("  🔍 %s", query)

            try:
                try:
                    raw = list(ddgs.text(
                        query,
                        backend="lite",
                        max_results=CONFIG["max_results"],
                        timelimit=CONFIG["time_filter"],
                    ))
                except TypeError:
                    # Fallback for older DDGS versions without timelimit
                    raw = list(ddgs.text(
                        query,
                        backend="lite",
                        max_results=CONFIG["max_results"],
                    ))
            except Exception as exc:
                log.warning("  DDGS error [%s]: %s", role, str(exc)[:80])
                time.sleep(5)
                continue

            if not raw:
                time.sleep(2)
                continue

            log.info("    📥 %d raw results — scraping …", len(raw))

            # Deduplicate before submitting to thread pool
            new_results = [r for r in raw if r.get("href", "") not in seen_urls]

            with ThreadPoolExecutor(max_workers=CONFIG["scrape_workers"]) as pool:
                futures = {
                    pool.submit(process_result, res, role): res
                    for res in new_results
                }
                try:
                    for future in as_completed(futures, timeout=90):
                        try:
                            job = future.result()
                            if job and job["apply_link"] not in seen_urls:
                                seen_urls.add(job["apply_link"])
                                records.append(job)
                        except Exception as exc:
                            log.debug("Future error: %s", str(exc)[:60])
                except FutureTimeout:
                    log.warning("  ⚠️  Some scrape futures timed out — continuing")

            for r in raw:
                seen_urls.add(r.get("href", ""))

            time.sleep(random.uniform(3.5, 6.0))  # Polite delay between sites

    log.info("  ✅ %s → %d valid jobs", role, len(records))
    return records


# ============================================================
# SECTION 8 — DELTA LAKE WRITER
# ============================================================

JOB_SCHEMA = StructType([
    StructField("id",                  StringType(),    True),
    StructField("job_hash",            StringType(),    True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("fetch_date",          DateType(),      True),
    StructField("search_keyword",      StringType(),    True),
    StructField("company_name",        StringType(),    True),
    StructField("job_title",           StringType(),    True),
    StructField("job_description",     StringType(),    True),
    StructField("apply_link",          StringType(),    True),
    StructField("hr_email",            StringType(),    True),
    StructField("job_type",            StringType(),    True),
    StructField("salary_range",        StringType(),    True),
    StructField("experience_required", StringType(),    True),
    StructField("location",            StringType(),    True),
    StructField("skills_required",     StringType(),    True),
    StructField("link_status",         StringType(),    True),
    StructField("validation_status",   StringType(),    True),
])

# New columns added in V6 (for schema-migration on existing tables)
_V6_NEW_COLUMNS = [
    ("hr_email",            "STRING"),
    ("job_type",            "STRING"),
    ("salary_range",        "STRING"),
    ("experience_required", "STRING"),
    ("location",            "STRING"),
    ("skills_required",     "STRING"),
    ("link_status",         "STRING"),
]


def ensure_table(spark: SparkSession, table: str):
    """Create catalog / schema / table if they don't exist, and add any missing V6 columns."""
    parts = table.split(".")
    if len(parts) == 3:
        catalog, schema, _ = parts
        for ddl in [
            f"CREATE CATALOG IF NOT EXISTS `{catalog}`",
            f"CREATE SCHEMA  IF NOT EXISTS `{catalog}`.`{schema}`",
        ]:
            try:
                spark.sql(ddl)
            except Exception:
                pass  # May lack permission if catalog already exists

    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {table} (
            id                  STRING,
            job_hash            STRING,
            created_at          TIMESTAMP,
            updated_at          TIMESTAMP,
            fetch_date          DATE,
            search_keyword      STRING,
            company_name        STRING,
            job_title           STRING,
            job_description     STRING,
            apply_link          STRING,
            hr_email            STRING,
            job_type            STRING,
            salary_range        STRING,
            experience_required STRING,
            location            STRING,
            skills_required     STRING,
            link_status         STRING,
            validation_status   STRING
        )
        USING DELTA
        PARTITIONED BY (fetch_date)
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact'   = 'true'
        )
    """)

    # Schema migration: add V6 columns to old tables that may be missing them
    try:
        existing_cols = {c.lower() for c in spark.table(table).columns}
        for col_name, col_type in _V6_NEW_COLUMNS:
            if col_name.lower() not in existing_cols:
                spark.sql(f"ALTER TABLE {table} ADD COLUMN {col_name} {col_type}")
                log.info("🔧 Schema migration — added column: %s", col_name)
    except Exception as exc:
        log.warning("Schema migration check failed (non-fatal): %s", exc)


def write_to_delta(records: list, spark: SparkSession) -> int:
    if not records:
        log.warning("No records to write.")
        return 0

    table = CONFIG["delta_table"]
    ensure_table(spark, table)

    df = spark.createDataFrame(records, schema=JOB_SCHEMA)
    df = df.dropDuplicates(["job_hash"])
    count = df.count()

    delta_tbl = DeltaTable.forName(spark, table)

    (
        delta_tbl.alias("tgt")
        .merge(df.alias("src"), "tgt.job_hash = src.job_hash")
        .whenMatchedUpdate(set={
            # Refresh mutable fields on re-run; keep created_at unchanged
            "updated_at":          "src.updated_at",
            "link_status":         "src.link_status",
            "hr_email":            "src.hr_email",
            "job_description":     "src.job_description",
            "job_type":            "src.job_type",
            "salary_range":        "src.salary_range",
            "skills_required":     "src.skills_required",
            "experience_required": "src.experience_required",
            "location":            "src.location",
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

    return count


# ============================================================
# SECTION 9 — MAIN ORCHESTRATOR
# ============================================================

def run_harvester():
    log.info("=" * 65)
    log.info("🚀 US IT JOB HARVESTER V6 — STARTING")
    log.info("📅 Run date  : %s", date.today())
    log.info("🎯 Roles     : %d", len(CONFIG["roles"]))
    log.info("📍 Locations : %d", len(CONFIG["locations"]))
    log.info("🔗 Portals   : %d", len(PORTAL_SITES))
    log.info("⏱️  Time filter: last-%s", {"d": "day", "w": "week", "m": "month"}.get(CONFIG["time_filter"], CONFIG["time_filter"]))
    log.info("=" * 65)

    all_records: list = []

    for role in CONFIG["roles"]:
        log.info("\n📌 Processing role: %s", role)
        try:
            role_jobs = search_for_role(role)
            all_records.extend(role_jobs)
        except Exception as exc:
            log.error("❌ Error for role '%s': %s", role, exc)
        time.sleep(random.uniform(4.0, 8.0))  # Polite gap between roles

    # Global dedup across all roles
    seen: set = set()
    unique: list = []
    for rec in all_records:
        h = rec["job_hash"]
        if h not in seen:
            seen.add(h)
            unique.append(rec)

    log.info("\n📊 Collected (all roles)  : %d", len(all_records))
    log.info("📊 After global dedup     : %d", len(unique))

    written = write_to_delta(unique, spark)

    log.info("\n" + "=" * 65)
    log.info("🎉 HARVEST COMPLETE!")
    log.info("   Total unique jobs written: %d", written)
    log.info("   Table: %s", CONFIG["delta_table"])
    log.info("=" * 65)

    # Quick summary printout (handy in notebook output)
    summary_df = spark.sql(f"""
        SELECT
            job_type,
            link_status,
            COUNT(*) AS jobs,
            COUNT(CASE WHEN hr_email != 'Not Found' THEN 1 END) AS with_email,
            COUNT(CASE WHEN salary_range != 'Not Specified' THEN 1 END) AS with_salary
        FROM {CONFIG['delta_table']}
        WHERE fetch_date = current_date()
        GROUP BY job_type, link_status
        ORDER BY jobs DESC
    """)
    log.info("\n📋 Today's summary:")
    summary_df.show(30, truncate=False)


# ============================================================
# ▶️  ENTRY POINT
# ============================================================
run_harvester()

In [0]:
%sql
select * from jobs_automation_db.default.raw_jobs_staging
-- delete from jobs_automation_db.default.raw_jobs_staging

In [0]:
import hashlib
import uuid
import re
import time
import random
import requests
from bs4 import BeautifulSoup
from datetime import date, datetime
from ddgs import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings("ignore", category=ResourceWarning)

# ==========================================
# 1. Strict Garbage Filter & Cleaner
# ==========================================
def is_garbage_link(url):
    # ట్యుటోరియల్స్, చైనీస్ సైట్స్, ఫోరమ్స్ ని ముందే కట్ చేస్తున్నాం
    bad_domains = [
        'wikipedia.org', 'w3schools.com', 'python.org', 'geeksforgeeks.org', 
        'taptap', 'zhihu', 'codecademy.com', 'cambridge.org', 'merriam-webster.com', 
        'datagov', 'ibm.com/docs', 'youtube.com', 'github.com/topics'
    ]
    return any(domain in url.lower() for domain in bad_domains)

def extract_title_and_company(raw_title, url):
    clean = re.sub(r'\[.*?\]', '', raw_title)
    clean = re.sub(r'\(.*?\)', '', clean)
    clean = re.sub(r'(?i)Job Application for\s+', '', clean)
    clean = re.sub(r'(?i)Jobs In\s+', ' at ', clean)
    
    # లింక్డ్ఇన్ మరియు ఇతర సైట్స్ చెత్తని తీయడం
    spam_suffixes = [
        ' - LinkedIn', ' | LinkedIn', ' - Greenhouse', ' - Lever', ' | Wellfound', 
        ' - Dice.com', ' | Built In'
    ]
    for suffix in spam_suffixes:
        clean = re.sub(rf'(?i){re.escape(suffix)}.*$', '', clean)
        
    company = "Web Extracted"
    job_title = clean
    
    split_chars = [' at ', ' At ', ' - ', ' | ', ' @ ']
    for char in split_chars:
        if char in clean:
            parts = clean.split(char, 1)
            job_title = parts[0].strip()
            company = parts[1].strip()
            break
            
    job_title = re.sub(r'[^a-zA-Z0-9\s\+\#\.]', '', job_title).strip()
    company = re.sub(r'[^a-zA-Z0-9\s]', '', company).strip()
    return job_title.title(), company.title()

# ==========================================
# 2. Smart Deep Link Scraper
# ==========================================
def extract_real_job_data(url, snippet):
    # 💡 FIX: లింక్డ్ఇన్ మరియు డైస్ (Dice) లాంటివి స్క్రాపింగ్ ని బ్లాక్ చేస్తాయి. 
    # కాబట్టి వాటికి సెర్చ్ ఇంజిన్ ఇచ్చే స్నిప్పెట్ ని మాత్రమే వాడతాం!
    if any(x in url.lower() for x in ["linkedin.com", "dice.com", "wellfound.com"]):
        return "", snippet
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
    }
    try:
        response = requests.get(url, headers=headers, timeout=8)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            for element in soup(["script", "style", "footer", "nav", "header"]):
                element.extract()
            page_title = soup.title.string if soup.title else ""
            text = soup.get_text(separator=' ', strip=True)
            clean_text = re.sub(r'\s+', ' ', text)
            
            # ఒకవేళ పేజీలో "Access Denied" లాంటివి వస్తే పాత స్నిప్పెట్ కే వెళ్లిపోతాం
            if "checking your browser" in clean_text.lower() or "access denied" in clean_text.lower():
                return "", snippet
                
            return page_title, clean_text[:4000]
    except Exception:
        pass
    return "", snippet

def process_search_result(res, role):
    apply_link = res.get('href', '')
    snippet = res.get('body', '')
    
    # 💡 GARBAGE FILTER: ముందే చెత్త లింక్స్ ని తీసేస్తాం
    if not apply_link or is_garbage_link(apply_link):
        return None 
        
    real_title, real_description = extract_real_job_data(apply_link, snippet)
    raw_title = real_title if real_title else res.get('title', '')
    
    job_title, company = extract_title_and_company(raw_title, apply_link)
    
    if not job_title or job_title == "":
        job_title = role
        
    job_hash = hashlib.md5(apply_link.encode('utf-8')).hexdigest()
    now = datetime.now()
    
    return {
        "id": str(uuid.uuid4()),
        "job_hash": job_hash,
        "created_at": now,
        "updated_at": now,
        "fetch_date": now.date(),
        "search_keyword": role,
        "company_name": company, 
        "job_title": job_title, 
        "job_description": real_description,
        "apply_link": apply_link,
        "validation_status": "Pending"
    }

# ==========================================
# 3. The Ultimate V5 Harvester (With US IT Portals)
# ==========================================
def execute_ultimate_harvester(roles_list, locations_list):
    print("🔥 Initiating THE ULTIMATE US-IT HARVESTER (V5)...")
    
    # 💡 FIX: నువ్వు చెప్పిన బెస్ట్ US IT మార్కెట్ పోర్టల్స్ ని యాడ్ చేశాను!
    target_sites = [
        "site:dice.com/job-detail",             # Heavyweight Tech
        "site:wellfound.com/jobs",              # Startups
        "site:weworkremotely.com/remote-jobs",  # Remote Only
        "site:remoteok.com",                    # Remote Only
        "site:greenhouse.io",                   # Direct ATS
        "site:jobs.lever.co",                   # Direct ATS
        "site:myworkdayjobs.com",               # Direct ATS
        "site:linkedin.com/jobs/view"           # LinkedIn
    ]
    
    locations_str = " OR ".join([f'"{loc}"' for loc in locations_list])
    records = []
    
    try:
        with DDGS() as ddgs:
            for role in roles_list:
                for site in target_sites:
                    # InURL keyword makes the search engine only look for job postings
                    query = f'{site} "{role}" ({locations_str})'
                    print(f"   🔍 Querying: {query}")
                    
                    try:
                        results = list(ddgs.text(query, backend='lite', max_results=15))
                        
                        if results:
                            print(f"      📥 Snatched {len(results)} links. Deep parsing securely...")
                            
                            with ThreadPoolExecutor(max_workers=4) as executor:
                                futures = [executor.submit(process_search_result, res, role) for res in results]
                                for future in as_completed(futures):
                                    job_data = future.result()
                                    if job_data:
                                        records.append(job_data)
                                        
                        time.sleep(random.uniform(2.5, 4.0))
                    except Exception:
                        time.sleep(5) 
                        
    except Exception as e:
        print(f"❌ Fatal Web Scraping Error: {str(e)}")
        
    if not records:
        print("\n🛑 No jobs found. Try again in an hour.")
        return
        
    print(f"\n✅ HARVEST COMPLETE! Grabbed {len(records)} ultra-clean IT links.")
    
    # ==========================================
    # 4. Delta Merge with STRICT SCHEMA
    # ==========================================
    job_schema = StructType([
        StructField("id", StringType(), True),
        StructField("job_hash", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True),
        StructField("fetch_date", DateType(), True),
        StructField("search_keyword", StringType(), True),
        StructField("company_name", StringType(), True),
        StructField("job_title", StringType(), True),
        StructField("job_description", StringType(), True),
        StructField("apply_link", StringType(), True),
        StructField("validation_status", StringType(), True)
    ])
    
    new_jobs_df = spark.createDataFrame(records, schema=job_schema)
    
    delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
    
    (delta_table.alias("target")
     .merge(
         new_jobs_df.alias("source"),
         "target.job_hash = source.job_hash"
     )
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"🎉 100% Success! {len(records)} jobs inserted into Staging securely.")

# 🚀 RUN THE ULTIMATE ENGINE
execute_ultimate_harvester(["Data Engineer", "Python Developer"], ["USA", "Remote", "Texas", "New York"])

In [0]:
import hashlib
import uuid
import re
import time
import random
import requests
from bs4 import BeautifulSoup
from datetime import date, datetime
from ddgs import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings("ignore", category=ResourceWarning)

# ==========================================
# 1. Strict Garbage Filter & Cleaner
# ==========================================
def is_garbage_link(url):
    # ట్యుటోరియల్స్, చైనీస్ సైట్స్, ఫోరమ్స్ ని ముందే కట్ చేస్తున్నాం
    bad_domains = [
        'wikipedia.org', 'w3schools.com', 'python.org', 'geeksforgeeks.org', 
        'taptap', 'zhihu', 'codecademy.com', 'cambridge.org', 'merriam-webster.com', 
        'datagov', 'ibm.com/docs', 'youtube.com', 'github.com/topics'
    ]
    return any(domain in url.lower() for domain in bad_domains)

def extract_title_and_company(raw_title, url):
    clean = re.sub(r'\[.*?\]', '', raw_title)
    clean = re.sub(r'\(.*?\)', '', clean)
    clean = re.sub(r'(?i)Job Application for\s+', '', clean)
    clean = re.sub(r'(?i)Jobs In\s+', ' at ', clean)
    
    # లింక్డ్ఇన్ మరియు ఇతర సైట్స్ చెత్తని తీయడం
    spam_suffixes = [
        ' - LinkedIn', ' | LinkedIn', ' - Greenhouse', ' - Lever', ' | Wellfound', 
        ' - Dice.com', ' | Built In'
    ]
    for suffix in spam_suffixes:
        clean = re.sub(rf'(?i){re.escape(suffix)}.*$', '', clean)
        
    company = "Web Extracted"
    job_title = clean
    
    split_chars = [' at ', ' At ', ' - ', ' | ', ' @ ']
    for char in split_chars:
        if char in clean:
            parts = clean.split(char, 1)
            job_title = parts[0].strip()
            company = parts[1].strip()
            break
            
    job_title = re.sub(r'[^a-zA-Z0-9\s\+\#\.]', '', job_title).strip()
    company = re.sub(r'[^a-zA-Z0-9\s]', '', company).strip()
    return job_title.title(), company.title()

# ==========================================
# 2. Smart Deep Link Scraper
# ==========================================
def extract_real_job_data(url, snippet):
    # 💡 FIX: లింక్డ్ఇన్ మరియు డైస్ (Dice) లాంటివి స్క్రాపింగ్ ని బ్లాక్ చేస్తాయి. 
    # కాబట్టి వాటికి సెర్చ్ ఇంజిన్ ఇచ్చే స్నిప్పెట్ ని మాత్రమే వాడతాం!
    if any(x in url.lower() for x in ["linkedin.com", "dice.com", "wellfound.com", "hiringcafe.com", ""]):
        return "", snippet
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8"
    }
    try:
        response = requests.get(url, headers=headers, timeout=8)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            for element in soup(["script", "style", "footer", "nav", "header"]):
                element.extract()
            page_title = soup.title.string if soup.title else ""
            text = soup.get_text(separator=' ', strip=True)
            clean_text = re.sub(r'\s+', ' ', text)
            
            # ఒకవేళ పేజీలో "Access Denied" లాంటివి వస్తే పాత స్నిప్పెట్ కే వెళ్లిపోతాం
            if "checking your browser" in clean_text.lower() or "access denied" in clean_text.lower():
                return "", snippet
                
            return page_title, clean_text[:4000]
    except Exception:
        pass
    return "", snippet

def process_search_result(res, role):
    apply_link = res.get('href', '')
    snippet = res.get('body', '')
    
    # 💡 GARBAGE FILTER: ముందే చెత్త లింక్స్ ని తీసేస్తాం
    if not apply_link or is_garbage_link(apply_link):
        return None 
        
    real_title, real_description = extract_real_job_data(apply_link, snippet)
    raw_title = real_title if real_title else res.get('title', '')
    
    job_title, company = extract_title_and_company(raw_title, apply_link)
    
    if not job_title or job_title == "":
        job_title = role
        
    job_hash = hashlib.md5(apply_link.encode('utf-8')).hexdigest()
    now = datetime.now()
    
    return {
        "id": str(uuid.uuid4()),
        "job_hash": job_hash,
        "created_at": now,
        "updated_at": now,
        "fetch_date": now.date(),
        "search_keyword": role,
        "company_name": company, 
        "job_title": job_title, 
        "job_description": real_description,
        "apply_link": apply_link,
        "validation_status": "Pending"
    }

# ==========================================
# 3. The Ultimate V5 Harvester (With US IT Portals)
# ==========================================
def execute_ultimate_harvester(roles_list, locations_list):
    print("🔥 Initiating THE ULTIMATE US-IT HARVESTER (V5)...")
    
    # 💡 FIX: నువ్వు చెప్పిన బెస్ట్ US IT మార్కెట్ పోర్టల్స్ ని యాడ్ చేశాను!
    target_sites = [
        "site:dice.com/job-detail",             # Heavyweight Tech
        "site:wellfound.com/jobs",              # Startups
        "site:weworkremotely.com/remote-jobs",  # Remote Only
        "site:remoteok.com",                    # Remote Only
        "site:greenhouse.io",                   # Direct ATS
        "site:jobs.lever.co",                   # Direct ATS
        "site:myworkdayjobs.com",               # Direct ATS
        "site:linkedin.com/jobs/view"           # LinkedIn
    ]
    
    locations_str = " OR ".join([f'"{loc}"' for loc in locations_list])
    records = []
    
    try:
        with DDGS() as ddgs:
            for role in roles_list:
                for site in target_sites:
                    # InURL keyword makes the search engine only look for job postings
                    query = f'{site} "{role}" ({locations_str})'
                    print(f"   🔍 Querying: {query}")
                    
                    try:
                        results = list(ddgs.text(query, backend='lite', max_results=15))
                        
                        if results:
                            print(f"      📥 Snatched {len(results)} links. Deep parsing securely...")
                            
                            with ThreadPoolExecutor(max_workers=4) as executor:
                                futures = [executor.submit(process_search_result, res, role) for res in results]
                                for future in as_completed(futures):
                                    job_data = future.result()
                                    if job_data:
                                        records.append(job_data)
                                        
                        time.sleep(random.uniform(2.5, 4.0))
                    except Exception:
                        time.sleep(5) 
                        
    except Exception as e:
        print(f"❌ Fatal Web Scraping Error: {str(e)}")
        
    if not records:
        print("\n🛑 No jobs found. Try again in an hour.")
        return
        
    print(f"\n✅ HARVEST COMPLETE! Grabbed {len(records)} ultra-clean IT links.")
    
    # ==========================================
    # 4. Delta Merge with STRICT SCHEMA
    # ==========================================
    job_schema = StructType([
        StructField("id", StringType(), True),
        StructField("job_hash", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True),
        StructField("fetch_date", DateType(), True),
        StructField("search_keyword", StringType(), True),
        StructField("company_name", StringType(), True),
        StructField("job_title", StringType(), True),
        StructField("job_description", StringType(), True),
        StructField("apply_link", StringType(), True),
        StructField("validation_status", StringType(), True)
    ])
    
    new_jobs_df = spark.createDataFrame(records, schema=job_schema)
    
    delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
    
    (delta_table.alias("target")
     .merge(
         new_jobs_df.alias("source"),
         "target.job_hash = source.job_hash"
     )
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"🎉 100% Success! {len(records)} jobs inserted into Staging securely.")

# 🚀 RUN THE ULTIMATE ENGINE
execute_ultimate_harvester(["Data Engineer", "Python Developer"], ["USA", "Remote", "Texas", "New York"])

In [0]:
import hashlib
import uuid
import re
import time
import random
import requests
from bs4 import BeautifulSoup
from datetime import date, datetime
from ddgs import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings("ignore", category=ResourceWarning)

# ==========================================
# 1. Smart Title & Company Extractor
# ==========================================
def extract_title_and_company(raw_title):
    # ATS సిస్టమ్స్ ఇచ్చే కామన్ చెత్త పదాలను తీసేస్తుంది
    clean_title = re.sub(r'(?i)Job Application for\s+', '', raw_title)
    clean_title = re.sub(r'(?i)Jobs In\s+', ' at ', clean_title)
    clean_title = re.sub(r'(?i)\s*-\s*Greenhouse.*$', '', clean_title)
    clean_title = re.sub(r'(?i)\s*-\s*Lever.*$', '', clean_title)
    clean_title = re.sub(r'(?i)\s*\|\s*LinkedIn.*$', '', clean_title)
    
    company = "Web Extracted"
    job_title = clean_title
    
    # ' at ', ' - ', ' | ' లాంటివి చూసి టైటిల్ ని కంపెనీని విడదీస్తుంది
    split_chars = [' at ', ' At ', ' - ', ' | ', ' @ ']
    for char in split_chars:
        if char in clean_title:
            parts = clean_title.split(char, 1)
            job_title = parts[0].strip()
            company = parts[1].strip()
            break
            
    # లాస్ట్ క్లీనప్ (Special characters తీసేయడానికి)
    job_title = re.sub(r'[^a-zA-Z0-9\s\+\#\.]', '', job_title).strip()
    company = re.sub(r'[^a-zA-Z0-9\s]', '', company).strip()
    
    return job_title.title(), company.title()

# ==========================================
# 2. Deep Link Scraper (With LinkedIn Bypass)
# ==========================================
def extract_real_job_data(url, snippet):
    # 💡 ANTI-CAPTCHA LOGIC: LinkedIn బాట్స్ ని బ్లాక్ చేస్తుంది కాబట్టి దాన్ని ఓపెన్ చేయము.
    # కేవలం స్నిప్పెట్ ని మాత్రమే వాడతాం.
    if "linkedin.com" in url:
        return "LinkedIn Job", snippet
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            for element in soup(["script", "style", "footer", "nav", "header"]):
                element.extract()
            page_title = soup.title.string if soup.title else ""
            text = soup.get_text(separator=' ', strip=True)
            clean_text = re.sub(r'\s+', ' ', text)
            return page_title, clean_text[:4500] # ఎగ్జాక్ట్ డిస్క్రిప్షన్!
    except Exception:
        pass
    return "", snippet

def process_search_result(res, role):
    apply_link = res.get('href', '')
    snippet = res.get('body', '')
    
    if not apply_link or "taptap" in apply_link.lower() or "zhihu" in apply_link.lower():
        return None 
        
    real_title, real_description = extract_real_job_data(apply_link, snippet)
    raw_title = real_title if real_title else res.get('title', '')
    
    # టైటిల్ ని కంపెనీని పర్ఫెక్ట్ గా లాగడం
    job_title, company = extract_title_and_company(raw_title)
    
    if not job_title or job_title == "":
        job_title = role
        
    job_hash = hashlib.md5(apply_link.encode('utf-8')).hexdigest()
    now = datetime.now()
    
    return {
        "id": str(uuid.uuid4()),
        "job_hash": job_hash,
        "created_at": now,
        "updated_at": None,
        "fetch_date": now.date(),
        "search_keyword": role,
        "company_name": company, 
        "job_title": job_title, 
        "job_description": real_description,
        "apply_link": apply_link,
        "validation_status": "Pending"
    }

# ==========================================
# 3. The Extreme Output Harvester Engine
# ==========================================
def execute_extreme_harvester(roles_list, locations_list):
    print("🔥 Initiating THE EXTREME HARVESTER (V4) Engine...")
    
    target_sites = [
        "site:greenhouse.io",
        "site:jobs.lever.co",
        "site:myworkdayjobs.com",
        "site:boards.ashbyhq.com",
        "site:linkedin.com/jobs/view" 
    ]
    
    locations_str = " OR ".join([f'"{loc}"' for loc in locations_list])
    records = []
    
    try:
        with DDGS() as ddgs:
            for role in roles_list:
                for site in target_sites:
                    query = f'{site} "{role}" ({locations_str})'
                    print(f"   🔍 Querying: {query}")
                    
                    try:
                        # 💡 max_results=20 పెట్టాం కాబట్టి ఒకేసారి చాలా జాబ్స్ వస్తాయి.
                        results = list(ddgs.text(query, backend='lite', max_results=20))
                        
                        if results:
                            print(f"      📥 Found {len(results)} links. Processing securely...")
                            
                            with ThreadPoolExecutor(max_workers=4) as executor:
                                futures = [executor.submit(process_search_result, res, role) for res in results]
                                for future in as_completed(futures):
                                    job_data = future.result()
                                    if job_data:
                                        records.append(job_data)
                                        
                        # సెర్చ్ కి సెర్చ్ కి మధ్య టైమ్ గ్యాప్ (యాంటీ-బాట్)
                        time.sleep(random.uniform(2.5, 4.5))
                    except Exception:
                        time.sleep(5) 
                        
    except Exception as e:
        print(f"❌ Fatal Web Scraping Error: {str(e)}")
        
    if not records:
        print("\n🛑 No jobs found. Try again in an hour.")
        return
        
    print(f"\n✅ EXTREME HARVEST COMPLETE! Grabbed {len(records)} clean links.")
    
    # ==========================================
    # 4. Delta Merge with New Schema
    # ==========================================
    # new_jobs_df = spark.createDataFrame(records)
    # ordered_columns = [
    #     "id", "job_hash", "created_at", "updated_at", "fetch_date", "search_keyword", 
    #     "company_name", "job_title", "job_description", "apply_link", "validation_status"
    # ]
    # new_jobs_df = new_jobs_df.select(*ordered_columns)

    # ==========================================
    # 4. Delta Merge with STRICT SCHEMA
    # ==========================================
    from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType

    # 💡 THE FIX: Explicitly telling Spark the exact data types!
    job_schema = StructType([
        StructField("id", StringType(), True),
        StructField("job_hash", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True),
        StructField("fetch_date", DateType(), True),
        StructField("search_keyword", StringType(), True),
        StructField("company_name", StringType(), True),
        StructField("job_title", StringType(), True),
        StructField("job_description", StringType(), True),
        StructField("apply_link", StringType(), True),
        StructField("validation_status", StringType(), True)
    ])
    
    # Passing the strict schema here solves the CANNOT_DETERMINE_TYPE error
    new_jobs_df = spark.createDataFrame(records, schema=job_schema)
    
    # We don't need to select ordered_columns anymore because the schema strictly enforces the order!
    
    delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
    
    (delta_table.alias("target")
     .merge(
         new_jobs_df.alias("source"),
         "target.job_hash = source.job_hash"
     )
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"🎉 100% Success! Clean, structured {len(records)} jobs merged into Staging without duplicates.")
    
    delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
    
    (delta_table.alias("target")
     .merge(
         new_jobs_df.alias("source"),
         "target.job_hash = source.job_hash"
     )
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"🎉 100% Success! Clean, structured jobs merged into Staging without duplicates.")

# 🚀 RUN THE EXTREME ENGINE
execute_extreme_harvester(["Data Engineer", "Python Developer"], ["USA", "Remote", "Texas", "New York", "California"])

In [0]:
%sql
select * from jobs_automation_db.default.raw_jobs_staging 

In [0]:
%sql
select * from jobs_automation_db.default.raw_jobs_staging 

In [0]:
import hashlib
import uuid
import re
import time
import random
import requests
from bs4 import BeautifulSoup
from datetime import date
from ddgs import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from concurrent.futures import ThreadPoolExecutor, as_completed
import warnings

warnings.filterwarnings("ignore", category=ResourceWarning)

# ==========================================
# 1. Advanced Job Title Cleaner
# ==========================================
def clean_job_title(raw_title):
    clean = re.sub(r'\[.*?\]', '', raw_title)
    clean = re.sub(r'\(.*?\)', '', clean)
    spam_words = ['remote', 'reputed company', 'entry level', 'w2', 'c2c', 'only', 'urgently hiring', 'urgent', 'hiring', 'new grads', 'usa', 'united states', 'contract', 'job application for', 'jobs at']
    for word in spam_words:
        clean = re.compile(rf'\b{re.escape(word)}\b', re.IGNORECASE).sub('', clean)
    clean = re.sub(r'[^a-zA-Z0-9\s\+]', '', clean)
    return " ".join(clean.split()).title()

# ==========================================
# 2. THE DEEP SCRAPER (Visits the actual URL)
# ==========================================
def extract_real_job_data(url):
    """URL ఓపెన్ చేసి అసలైన జాబ్ డిస్క్రిప్షన్ ని లాగుతుంది"""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'lxml')
            
            # Remove scripts, styles, footers, headers
            for element in soup(["script", "style", "footer", "nav", "header"]):
                element.extract()
            
            # Extract real title from HTML
            page_title = soup.title.string if soup.title else ""
            
            # Extract main text
            text = soup.get_text(separator=' ', strip=True)
            
            # Reduce massive whitespaces
            clean_text = re.sub(r'\s+', ' ', text)
            
            # Return up to 4000 chars (plenty for AI to read)
            return page_title, clean_text[:4000]
    except Exception as e:
        print(f"      ⚠️ Failed to load page deeply: {url} - {str(e)}")
        
    return "", ""

# ==========================================
# 3. Process Single Query (Threaded Function)
# ==========================================
def process_search_result(res, role):
    apply_link = res.get('href', '')
    if not apply_link or "taptap" in apply_link.lower() or "zhihu" in apply_link.lower():
        return None # Chinese gaming/forum links filter out
    
    # 1. VISITING THE LINK DIRECTLY!
    real_title, real_description = extract_real_job_data(apply_link)
    
    # If the page failed to load, fallback to search engine title
    raw_title = real_title if real_title else res.get('title', '')
    raw_title = raw_title.replace(' - LinkedIn', '')
    
    cleaned_title = clean_job_title(raw_title)
    
    # 2. HASH GENERATION
    job_hash = hashlib.md5(apply_link.encode('utf-8')).hexdigest()
    
    return {
        "job_hash": job_hash,
        "raw_job_id": str(uuid.uuid4()),
        "fetch_date": date.today(),
        "search_keyword": role,
        "company_name": "Deep Web Extracted", 
        "job_title": cleaned_title, 
        "job_description": real_description if real_description else res.get('body', ''), # Saving Real description!
        "apply_link": apply_link,
        "validation_status": "Pending"
    }

# ==========================================
# 4. Ultimate Deep Harvester Engine
# ==========================================
def fetch_jobs_deep_stealth(roles_list, locations_list):
    print("🥷 Initiating DEEP STEALTH Web Harvester (URL Visiting Mode)...")
    
    target_sites = [
        "site:greenhouse.io",
        "site:jobs.lever.co",
        "site:linkedin.com/jobs/view"
    ]
    
    locations_str = " OR ".join([f'"{loc}"' for loc in locations_list])
    records = []
    
    try:
        with DDGS() as ddgs:
            for role in roles_list:
                for site in target_sites:
                    query = f'{site} "{role}" ({locations_str})'
                    print(f"   🔍 Stealth Searching: {query}")
                    
                    try:
                        results = list(ddgs.text(query, backend='html', max_results=5))
                        
                        if results:
                            print(f"      📥 Found {len(results)} links. Opening URLs deeply to fetch real descriptions...")
                            
                            # Use Threading to visit multiple URLs at the same time
                            with ThreadPoolExecutor(max_workers=3) as executor:
                                futures = [executor.submit(process_search_result, res, role) for res in results]
                                
                                for future in as_completed(futures):
                                    job_data = future.result()
                                    if job_data:
                                        records.append(job_data)
                                        
                        time.sleep(random.uniform(2.0, 4.0))
                        
                    except Exception as inner_e:
                        print(f"      ⚠️ Search engine pushed back. Sleeping...")
                        time.sleep(5)

    except Exception as e:
        print(f"❌ Fatal Web Scraping Error: {str(e)}")
        
    if not records:
        print("\n🛑 No jobs found or blocked.")
        return
        
    print(f"\n✅ Boom! Sneakily visited {len(records)} links and grabbed real descriptions. Merging into Databricks...")
    
    new_jobs_df = spark.createDataFrame(records)
    ordered_columns = [
        "job_hash", "raw_job_id", "fetch_date", "search_keyword", 
        "company_name", "job_title", "job_description", "apply_link", "validation_status"
    ]
    new_jobs_df = new_jobs_df.select(*ordered_columns)
    
    delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
    
    (delta_table.alias("target")
     .merge(
         new_jobs_df.alias("source"),
         "target.job_hash = source.job_hash"
     )
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"🎉 Success! Harvester Complete. Unique, Deeply Scraped jobs added to Staging.")

# ==========================================
# 🚀 RUNNING THE DEEP HARVESTER 
# ==========================================
roles = ["Data Engineer", "Python Developer"]
locations = ["Remote USA", "Texas", "New York"]

fetch_jobs_deep_stealth(roles, locations)

In [0]:
%sql
select * from jobs_automation_db.default.raw_jobs_staging where company_name = "Deep Web Extracted"

In [0]:
import hashlib
import uuid
import re
import time
import random
import warnings
from datetime import date
from ddgs import DDGS
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

warnings.filterwarnings("ignore", category=ResourceWarning)

# ==========================================
# 1. Advanced Job Title Cleaner
# ==========================================
def clean_job_title(raw_title):
    clean = re.sub(r'\[.*?\]', '', raw_title)
    clean = re.sub(r'\(.*?\)', '', clean)
    spam_words = ['remote', 'reputed company', 'entry level', 'w2', 'c2c', 'only', 'urgently hiring', 'urgent', 'hiring', 'new grads', 'usa', 'united states', 'contract']
    for word in spam_words:
        clean = re.compile(rf'\b{re.escape(word)}\b', re.IGNORECASE).sub('', clean)
    clean = re.sub(r'[^a-zA-Z0-9\s]', '', clean)
    return " ".join(clean.split()).title()

# ==========================================
# 2. Stealth Web Harvester (Anti-Block)
# ==========================================
def fetch_jobs_stealth_mode(roles_list, locations_list):
    print("🥷 Initiating STEALTH Web Harvester (Bypassing IP Blocks)...")
    
    target_sites = [
        "site:linkedin.com/jobs/view",
        "site:greenhouse.io",
        "site:jobs.lever.co"
    ]
    
    # 💡 FIX 1: Grouping locations to reduce number of queries!
    locations_str = " OR ".join([f'"{loc}"' for loc in locations_list])
    records = []
    
    try:
        with DDGS() as ddgs:
            for role in roles_list:
                for site in target_sites:
                    # e.g., site:greenhouse.io "Data Engineer" ("Remote" OR "Texas" OR "New York")
                    query = f'{site} "{role}" ({locations_str})'
                    print(f"   🔍 Stealth Surfing: {query}")
                    
                    try:
                        # 💡 FIX 2: backend='html' avoids the strict API limits of duckduckgo
                        results = list(ddgs.text(query, backend='html', max_results=10))
                        
                        if results:
                            for res in results:
                                raw_title = res.get('title', '').replace(' - LinkedIn', '')
                                cleaned_title = clean_job_title(raw_title)
                                apply_link = res.get('href', '')
                                
                                job_hash = hashlib.md5(apply_link.encode('utf-8')).hexdigest()
                                
                                records.append({
                                    "job_hash": job_hash,
                                    "raw_job_id": str(uuid.uuid4()),
                                    "fetch_date": date.today(),
                                    "search_keyword": role,
                                    "company_name": "Web Extracted", 
                                    "job_title": cleaned_title, 
                                    "job_description": res.get('body', ''),
                                    "apply_link": apply_link,
                                    "validation_status": "Pending"
                                })
                        else:
                            print("      ⚠️ No results found for this specific query.")
                        
                        # 💡 FIX 3: Random sleep between 3 to 6 seconds to mimic human behavior!
                        sleep_time = random.uniform(3.0, 6.0)
                        time.sleep(sleep_time)
                        
                    except Exception as inner_e:
                        print(f"      ⚠️ Search engine pushed back: {str(inner_e)}")
                        # If blocked, wait longer!
                        time.sleep(10)

    except Exception as e:
        print(f"❌ Fatal Web Scraping Error: {str(e)}")
        
    if not records:
        print("\n🛑 Still Blocked! Databricks IP is heavily flagged today. We need a Hybrid approach.")
        return
        
    print(f"\n📥 Boom! Sneakily snatched {len(records)} links. Merging into Databricks...")
    
    # ==========================================
    # 3. Delta Merge
    # ==========================================
    new_jobs_df = spark.createDataFrame(records)
    ordered_columns = [
        "job_hash", "raw_job_id", "fetch_date", "search_keyword", 
        "company_name", "job_title", "job_description", "apply_link", "validation_status"
    ]
    new_jobs_df = new_jobs_df.select(*ordered_columns)
    
    delta_table = DeltaTable.forName(spark, "jobs_automation_db.default.raw_jobs_staging")
    
    (delta_table.alias("target")
     .merge(
         new_jobs_df.alias("source"),
         "target.job_hash = source.job_hash"
     )
     .whenNotMatchedInsertAll()
     .execute())
    
    print(f"✅ Success! Harvester Complete. Only UNIQUE and NEW jobs were added to Staging.")

# ==========================================
# 🚀 RUNNING THE STEALTH HARVESTER 
# ==========================================
roles = ["Data Engineer", "Python Developer"]
locations = ["Remote", "Texas", "New York"]

fetch_jobs_stealth_mode(roles, locations)

In [0]:
%sql
select * from jobs_automation_db.default.raw_jobs_staging where company_name = "Web Extracted"

In [0]:
%sql
select * from jobs_automation_db.default.raw_jobs_staging

In [0]:
import json
import requests
import re
import uuid
from datetime import date, datetime
from pyspark.sql import SparkSession
from ddgs import DDGS
from concurrent.futures import ThreadPoolExecutor, as_completed

# API Key for AI Validation
NVIDIA_NIM_API_KEY = dbutils.secrets.get("jobs_automation", "nvidia_nemotron_3_ultra")

# ==========================================
# 1. LIVE WEB JOB HUNTER (Bypassing JSearch)
# ==========================================
def fetch_fresh_jobs_from_web(role, location="Remote"):
    print(f"🌐 Surfing the web directly for '{role}' jobs posted in the LAST 24 HOURS...")
    
    # We target specific ATS platforms and LinkedIn to avoid garbage links
    queries = [
        f'site:linkedin.com/jobs/view "{role}" "{location}"',
        f'site:greenhouse.io "{role}" "{location}"',
        f'site:jobs.lever.co "{role}" "{location}"'
    ]
    
    fresh_jobs = []
    
    try:
        with DDGS() as ddgs:
            for q in queries:
                print(f"   🔍 Querying: {q}")
                # timelimit='d' means strictly PAST 24 HOURS! 'w' is for past week.
                results = list(ddgs.text(q, timelimit='d', max_results=10))
                
                for res in results:
                    # Clean the title from standard LinkedIn suffixes
                    title_clean = res.get('title', '').replace(' - LinkedIn', '')
                    
                    fresh_jobs.append({
                        "raw_job_id": str(uuid.uuid4()),
                        "fetch_date": date.today(),
                        "search_keyword": role,
                        "company_name": "Web/LinkedIn Job", # Will be updated by AI later
                        "job_title": title_clean,
                        "job_description": res.get('body', ''), # The search snippet
                        "apply_link": res.get('href', ''),
                        "validation_status": "Pending"
                    })
    except Exception as e:
        print(f"❌ Web Scraping Failed: {str(e)}")
        
    return fresh_jobs

# ==========================================
# 2. BULLETPROOF AI PARSER (Fixing the JSON Error)
# ==========================================
def validate_job_with_ai(job_title, job_snippet, apply_link):
    url = "https://integrate.api.nvidia.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {NVIDIA_NIM_API_KEY}",
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    You are an expert Bench Sales Recruiter.
    Analyze this freshly scraped job snippet from the web:
    
    Job Title: {job_title}
    Snippet: {job_snippet}
    URL: {apply_link}
    
    Instructions:
    1. Extract the actual Company Name from the Title, Snippet, or URL. If you can't find it, use "Direct Client".
    2. Determine if it is a real IT job requirement (True) or just a random non-job page (False).
    3. Determine the Job Type: [W2, C2C, Contract, Full-Time, Unknown].
    4. Extract any HR email if visible.
    
    Return ONLY a valid JSON object matching exactly this structure:
    {{
        "company_name": "Extracted Company Name",
        "is_valid": true,
        "job_type": "Contract/C2C",
        "hr_email": "hr@company.com or null",
        "reasoning": "Clear IT job posted recently."
    }}
    """
    
    payload = {
        "model": "meta/llama-3.1-70b-instruct", 
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 500
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload)
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content'].strip()
            
            # 💡 FIX: Ultra-robust JSON Extraction using Regex
            match = re.search(r'\{.*\}', content, re.DOTALL)
            if match:
                try:
                    return json.loads(match.group(0))
                except json.JSONDecodeError:
                    # If strict JSON fails, try cleaning bad quotes or trailing commas
                    cleaned = match.group(0).replace("'", '"').replace(",}", "}")
                    return json.loads(cleaned)
        return None
    except Exception as e:
        return None

# ==========================================
# 3. Process Execution (Fetch -> Validate -> Store)
# ==========================================
def execute_direct_web_pipeline(search_role):
    # 1. Fetch exactly fresh jobs from today
    fresh_jobs = fetch_fresh_jobs_from_web(search_role)
    
    if not fresh_jobs:
        print("🛑 No fresh jobs found in the last 24 hours for this query.")
        return
        
    print(f"\n✅ Successfully snatched {len(fresh_jobs)} FRESH jobs from LinkedIn/Greenhouse/Lever!")
    print("🧠 Passing them to AI for validation and extraction...\n")
    
    validated_records = []
    
    # 2. Multi-threaded AI Validation
    with ThreadPoolExecutor(max_workers=3) as executor:
        future_to_job = {executor.submit(validate_job_with_ai, job['job_title'], job['job_description'], job['apply_link']): job for job in fresh_jobs}
        
        for future in as_completed(future_to_job):
            job = future_to_job[future]
            ai_result = future.result()
            
            if ai_result and ai_result.get("is_valid"):
                comp_name = ai_result.get("company_name", "Unknown")
                print(f"   ✔️ VALID FRESH JOB! {job['job_title']} @ {comp_name}")
                
                validated_records.append({
                    "job_id": job['raw_job_id'],
                    "company_name": comp_name,
                    "job_title": job['job_title'],
                    "job_type": ai_result.get("job_type", "Unknown"),
                    "is_valid": True,
                    "hr_email": ai_result.get("hr_email"),
                    "apply_url": job['apply_link'],
                    "validated_date": datetime.now()
                })
            else:
                print(f"   ❌ REJECTED (Not a valid job posting): {job['apply_link']}")

    # 3. Store valid jobs directly to our Master Table
    if len(validated_records) > 0:
        valid_df = spark.createDataFrame(validated_records)
        ordered_columns = [
            "job_id", "company_name", "job_title", "job_type", 
            "is_valid", "hr_email", "apply_url", "validated_date"
        ]
        valid_df = valid_df.select(*ordered_columns)
        
        # Append to Master
        valid_df.write.mode("append").insertInto("jobs_automation_db.default.validated_jobs_master")
        print(f"\n💾 BOOM! Saved {len(validated_records)} ULTRA-FRESH GOLDEN jobs to master table.")

# RUN THE NEW PIPELINE
execute_direct_web_pipeline("Data Engineer")